# esmDMS Simulation!!

Benchmarking out inference methods with perfect data!

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib import rc
from numpy.linalg import inv 
from sklearn import linear_model 
import stan
from esmdmsfunctions import *

In [2]:
rc('text', usetex=True)
pd.set_option('display.max_columns', 100)
pwd = os.getcwd()

#init_pop = pd.read_pickle(pwd + '/data/sequence_data/all_reps_BF520_protein_embeddings.pkl')

#init_pop = init_pop[init_pop['Embeddings'].notna()].reset_index(drop=True)

In [3]:

# Now, let's calculate the fitness for each variant based on its embedding and the selection coefficients
def calculate_fitness_exp(embedding, selection_coefficients):
    fitness = np.exp(np.dot(embedding, selection_coefficients))
    if np.isinf(fitness):
        fitness = 1e10  # Cap infinite fitness to a large number
    return fitness

def calc_all_fitness_exp(embeddings, selection_coefficients,
                         embedding_clip=None):
    fitnesses = []
    for embedding in embeddings:
        if embedding_clip is not None:
            embedding = np.clip(embedding, embedding_clip[0], embedding_clip[1])
        fitness = calculate_fitness_exp(embedding, selection_coefficients)
        fitnesses.append(fitness)
    return np.array(fitnesses)

def calculate_fitness_plus1(embedidng, selection_coefficients):
    fitness = 1 + np.dot(embedidng, selection_coefficients)
    return max(fitness, 0)  # Ensure fitness is not negative

def calc_all_fitness_plus1(embeddings, selection_coefficients,
                            embedding_clip=None):
    fitnesses = []
    for embedding in embeddings:
        if embedding_clip is not None:
            embedding = np.clip(embedding, embedding_clip[0], embedding_clip[1])
        fitness = calculate_fitness_plus1(embedding, selection_coefficients)
        fitnesses.append(fitness)
    return np.array(fitnesses)

def simulate_generation_multinomial(current_counts, fitnesses):
    population_size = np.sum(current_counts)
    total_fitness = np.sum(current_counts * fitnesses)
    probabilities = (current_counts * fitnesses) / total_fitness
    # if a probability is below 0, set it to zero
    # if a probability is above 1, set it to 1
    probabilities = np.clip(probabilities, 0, 1)
    next_counts = np.random.multinomial(population_size, probabilities)
    return next_counts # Check if output is a different scale


In [ ]:

def get_df_selection(random_seed=42, selected_layer=12, normalize_embeddings=True, input_df=None):
    """Get the starting information for the simulation,
    i.e. the layer dataframe and the initial counts for each replicate.
    
    Args:
        input_df: Optional pre-built dataframe containing Rep1/2/3_PreNums,
                  Rep1/2/3_PostNums, and an 'Embedding' column for the selected
                  layer. If provided, data loading and layer decomposition are
                  skipped entirely.
    """
    rc('text', usetex=True)
    pd.set_option('display.max_columns', 100)
    np.random.seed(random_seed)

    if input_df is not None:
        # Expect input_df to already have Rep{1,2,3}_Pre/PostNums and 'Embedding'
        required_cols = [
            'Rep1_PreNums', 'Rep2_PreNums', 'Rep3_PreNums',
            'Rep1_PostNums', 'Rep2_PostNums', 'Rep3_PostNums',
            'Embedding'
        ]
        missing = [c for c in required_cols if c not in input_df.columns]
        if missing:
            raise ValueError(f"input_df is missing required columns: {missing}")
        df_selection = input_df.copy()

    else:
        pwd = os.getcwd()
        init_pop = get_unique_df(pwd + '/data/sequence_data/all_reps_BF520_protein_embeddings.pkl')
        n_reps = 3

        replicate_dfs = []
        for rep in range(n_reps):
            df_rep = init_pop.copy()
            df_rep['PreNums']  = df_rep['PreNums'].apply(lambda x: x[rep])
            df_rep['PostNums'] = df_rep['PostNums'].apply(lambda x: x[rep])
            df_rep = df_rep.drop(columns=['ProteinSequence'])
            replicate_dfs.append(df_rep)

        # Only decompose the selected layer — skip the full loop
        decomposed_dfs = []
        for rep_df in replicate_dfs:
            decomposed_df = rep_df[['PreNums', 'PostNums']].copy()
            decomposed_df[f'Layer{selected_layer}'] = rep_df['Embeddings'].apply(
                lambda x: x[selected_layer]
            )
            decomposed_dfs.append(decomposed_df)

        df_selection = pd.DataFrame()
        for i, decomposed_df in enumerate(decomposed_dfs):
            df_selection[f'Rep{i+1}_PreNums']  = decomposed_df['PreNums']
            df_selection[f'Rep{i+1}_PostNums'] = decomposed_df['PostNums']
        df_selection['Embedding'] = decomposed_dfs[-1][f'Layer{selected_layer}']

    if normalize_embeddings:
        embeddings   = np.vstack(df_selection['Embedding'].tolist())
        z_embeddings = z_normalize(embeddings)
        df_selection['Embedding'] = [z_embeddings[i] for i in range(z_embeddings.shape[0])]

    start_counts1 = df_selection["Rep1_PreNums"].values
    start_counts2 = df_selection["Rep2_PreNums"].values
    start_counts3 = df_selection["Rep3_PreNums"].values

    return df_selection, (start_counts1, start_counts2, start_counts3)


def run_simulation(df_selection, selection_coefficients, initial_counts, 
                   n_gens=30, embedding_clip=None, save_every=1, fitness='exp'):
    embeddings = np.vstack(df_selection['Embedding'].values)
    
    if fitness == 'exp':
        fitnesses = calc_all_fitness_exp(embeddings, selection_coefficients,
                                        embedding_clip=embedding_clip)
    elif fitness == 'plus1':
        fitnesses = calc_all_fitness_plus1(embeddings, selection_coefficients,
                                        embedding_clip=embedding_clip)
    else:
        raise ValueError("Invalid fitness function specified.")
    
    #print("intial counts shape and type:", np.array(initial_counts).shape, type(initial_counts))
    generation_counts = [initial_counts]
    n_reps = len(initial_counts)
    last_counts = initial_counts
    
    print(last_counts)
    
    for gen in range(n_gens):
        
        this_gen = []
        for rep in range(n_reps):
            rep_counts = last_counts[rep]
            next_counts = simulate_generation_multinomial(rep_counts, fitnesses)
            this_gen.append(next_counts)
        if (gen + 1) % save_every == 0:
            #print(f"Completed generation {gen + 1}")
            generation_counts.append(this_gen)
        last_counts = this_gen
    #print(f"generation_counts length: {len(generation_counts)}")
    #print(f"generation_counts shape: {np.array(generation_counts).shape}")
    return generation_counts, fitnesses

In [5]:
# Now, calculate selectiion_coefficitents from the generational counts

import popDMS
import pandas as pd
import pickle
from importlib import reload
# import the pearsonr function
from scipy.stats import pearsonr

# reload popDMS
reload(popDMS)


def simulation_df_transfer(df_selection, generation_counts):
    #["Generation", "Embedding", "Frequency", "Replicate"]
    emb_vals = df_selection["Embedding"].values
    #generation counts format [[gen1data], [gen2data],...]
    data_list = []
    for gen, gen_data in enumerate(generation_counts):
        for rep, rep_data in enumerate(gen_data):
            for i, count in enumerate(rep_data):
                data_list.append({
                    "Generation": gen,
                    "Embedding": emb_vals[i],
                    "Frequency": count,
                    "Replicate": rep + 1
                })
    return pd.DataFrame(data_list)

def run_inference_calcs_sims(df_selection, generation_counts, output_path,
                             save_output=False):
    """Run the inference calculations:
    WHOLE PIPELINE FROM READING IN EMBEDDINGS DATAFRAME
    """
    inference_df = simulation_df_transfer(df_selection, generation_counts)
    
    if save_output:
        # make directory
        if not os.path.exists(output_path):
            os.makedirs(output_path)
        # save inference_df
        inference_df.to_pickle(output_path + 'inference_df.pkl')
        print(f"SAVED INFERENCE DF TO {output_path}")
    
       
    data = popDMS.mini_infer_independent_esm(inference_df, n_replicates=3)
    return data # data = [dx, icov, s, s_joint, sel_data, gamma_opt, x_array]


In [ ]:
# Simulate other methods

def other_methods(df_selection, generation_counts, generation=-1):
    """Find the enrichment ratio, log ratio, and log enrichment"""
    first_gen = generation_counts[0]
    last_gen = generation_counts[generation]
    
    enrichments = []
    log_ratios = []
    #log_enrichments = []
    
    embeddings = np.vstack(df_selection['Embedding'].values)
    
    for rep in range(len(first_gen)):
        start_counts = first_gen[rep]
        end_counts = last_gen[rep]
        
        embedding_avg_before = np.sum(embeddings.T * start_counts, axis=1) / np.sum(start_counts)
        embedding_avg_after = np.sum(embeddings.T * end_counts, axis=1) / np.sum(end_counts)
        
        
        embedding_sum = np.sum(embedding_avg_before) + np.sum(embedding_avg_after) / 2
        
        enrichment = (embedding_avg_after / embedding_avg_before) / embedding_sum
        
        log_ratio = np.log((embedding_avg_after / embedding_avg_before) / embedding_sum)
        
        #log_enrichment = np.log(enrichment)
        
        log_ratios.append(log_ratio)
        enrichments.append(enrichment)
        
    return np.array(enrichments), np.array(log_ratios) #, np.array(log_enrichments)

def normalize_embeddings(df):
    #print(df.head())
    embedding_array = np.array([x for x in df["Embedding"].to_list()])
    #print(embedding_array.shape)
    z_embeddings = np.zeros_like(embedding_array)
    dimensions = embedding_array.shape[1]
    for dim in range(dimensions):
        z_embeddings[:, dim] = z_normalize(embedding_array[:, dim])
        
    # insert back into the dataframe
    df["Embedding"] = [z_embeddings[i] for i in range(z_embeddings.shape[0])]
    return df


def generate_selection(embeddings):
    embedding_ranges = embeddings.max(axis=0) - embeddings.min(axis=0)
    selection_coefficients = np.zeros(embeddings.shape[1])
    # Find the indices of the top, lowest, and middle range dimensions
    sorted_indices = np.argsort(embedding_ranges)
    high_range_idx = sorted_indices[-1]
    # Give these dimensions higher selection coefficients
    selection_coefficients[high_range_idx] = 0.10
    return selection_coefficients

def generate_one_selection(embeddings):
    selection_coefficients = np.zeros(embeddings.shape[1])
    embedding_ranges = embeddings.max(axis=0) - embeddings.min(axis=0)
    # Find the indices of the top, lowest, and middle range dimensions
    sorted_indices = np.argsort(embedding_ranges)
    high_range_idx = sorted_indices[-1]
    # Give these dimensions higher selection coefficients
    selection_coefficients[high_range_idx] = 0.10
    return selection_coefficients

def get_simulation_results(n_layers, n_gens, n_reps, 
                           sel_func=generate_selection,
                           inference=True, fitness='plus1',
                           save_every=1):
    all_layer_fits = {}
    all_selection_coefficients = {}
    detailed_selection_results = {}
    all_generation_counts = {}
    whole_embedding_matrix = {}
    for layer in range(n_layers):
        print(f"Running layer {layer}...")
        df_selection, initial_counts = get_df_selection(random_seed=42, selected_layer=layer,
                                                        normalize_embeddings=False)
        print(df_selection.head())
        df_selection = normalize_embeddings(df_selection)

        # Generate selection coefficients using the provided function
        embedding_matrix = np.vstack(df_selection['Embedding'].tolist())
        whole_embedding_matrix[layer] = embedding_matrix
        selection_coefficients = sel_func(embedding_matrix)
        all_selection_coefficients[layer] = selection_coefficients
        
        print("Running simulation...")
        generation_counts, layer_fits = run_simulation(df_selection, selection_coefficients, 
                                           initial_counts, n_gens=n_gens, save_every=save_every,
                                           fitness=fitness)
        all_generation_counts[layer] = generation_counts
        all_layer_fits[layer] = layer_fits

        if inference:
            gen=n_gens
            print(f"  Analyzing generation {gen}...")
            layer_results = []
            test_path = pwd + f"/simulations/layer_{layer}_gen_{gen}/"
            data = run_inference_calcs_sims(df_selection, generation_counts[:gen + 1], test_path)
            found_sel_coeffs = data[2]
            layer_results.append(found_sel_coeffs)

            detailed_selection_results[layer] = layer_results

    return all_layer_fits, all_selection_coefficients, detailed_selection_results, all_generation_counts, whole_embedding_matrix
    
    
    
def calc_inferred_fits(sim_data, fitness='plus1', layer=0):
    """ Calculate the inferred fitness score of every individual in the population across layer and generation using the inferred selection coefficients and the embeddings"""
    sel_coefs = sim_data[2][layer][0]
    embeddings = sim_data[4][layer]
    n_reps = sel_coefs.shape[0]
    n_indivs = embeddings.shape[0]
    inferred_fits = []
    for indiv in range(n_indivs):
        indiv_fits = []
        for rep in range(n_reps):
            if fitness == 'exp':
                fit = calculate_fitness_exp(embeddings[indiv], sel_coefs[rep])
            elif fitness == 'plus1':
                fit = calculate_fitness_plus1(embeddings[indiv], sel_coefs[rep])
            else:
                raise ValueError("Invalid fitness function specified.")
            indiv_fits.append(fit)
        inferred_fits.append(indiv_fits)
    return np.array(inferred_fits)
    
def comp_inf_vs_real_fits(sim_data, layer, fitness='plus1'):
    """Compare the inferred and real fitness scores."""
    # For each layer, plot the fitness growth over time
    all_layer_fits = sim_data[0][layer]
    all_gen_counts = sim_data[3]
    embeddings = sim_data[4][layer]
    n_gens = len(all_gen_counts[0]) - 1
    n_reps = len(all_gen_counts[0][0])

    # Print the shape of all these data
    #print(fitness)
    inferred_fits = calc_inferred_fits(sim_data, fitness=fitness, 
                                       layer=layer)

    # Z-normalize the real fits
    real_fits = np.array(all_layer_fits)
    real_fits = z_normalize(real_fits)

    # Normalize the inferred fits per replicate
    rep_fits = {}
    for rep in range(n_reps):
        rep_fits[rep] = z_normalize(inferred_fits[:, rep])

    # Make a plot showing the comparison between the real and inferred
    # fitness scores for each replicate, with a diagonal line for reference
    fig, axes = plt.subplots(1, n_reps, figsize=(6 * n_reps, 6), squeeze=False)
    axes = axes.flatten()
    plt.style.use('seaborn-v0_8-darkgrid')

    all_real = []
    all_inferred = []

    for rep in range(n_reps):
        ax = axes[rep]
        r = real_fits
        inf = rep_fits[rep]
        ax.scatter(r, inf, alpha=0.5)
        ax.set_xlabel('Real Fitness (Normalized)')
        ax.set_ylabel('Inferred Fitness (Normalized)')
        ax.set_title(f'Layer {layer} — Replicate {rep}')

        # Diagonal reference line
        lims = [min(r.min(), inf.min()) - 0.5,
                max(r.max(), inf.max()) + 0.5]
        ax.plot(lims, lims, color='red', linestyle='--')
        ax.set_xlim(lims)
        ax.set_ylim(lims)

        # Compute and annotate Pearson r
        
        
        corr, pval = pearsonr(r, inf)
        ax.annotate(f'r = {corr:.3f}\np = {pval:.2e}',
                     xy=(0.05, 0.95), xycoords='axes fraction',
                     ha='left', va='top',
                     fontsize=11, bbox=dict(boxstyle='round', fc='white', alpha=0.8))

        all_real.extend(r)
        all_inferred.extend(inf)

    plt.suptitle(f'Real vs Inferred Fitness — Layer {layer}', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

    # Also return the overall correlation across all replicates
    overall_corr, overall_pval = pearsonr(all_real, all_inferred)
    print(f'Overall Pearson r = {overall_corr:.4f}, p = {overall_pval:.2e}')

    return rep_fits, overall_corr



def find_fixed_gen(generation_counts, cutoff_pct=0.75):
    # Return the generation at which 90% of the population has the same dominant type
    for gen in range(len(generation_counts)):
        for rep in range(len(generation_counts[gen])):
            rep_counts = generation_counts[gen][rep]
            total_count = np.sum(rep_counts)
            max_count = np.max(rep_counts)
            #print(f"Generation: {gen} Replicate: {rep} Max Count: {max_count} Total Count: {total_count}")
            if max_count / total_count >= cutoff_pct:
                return gen
    return len(generation_counts) - 1  # Return the last generation if never reaches cutoff

def averaged_covariance(sim_data, layer=0):
    layer_df = get_df_selection(random_seed=42, selected_layer=layer,
                                    normalize_embeddings=False)[0]
    layer_df = normalize_embeddings(layer_df)
    embeddings = np.vstack(layer_df['Embedding'].tolist())

    # Get the selection coefficients
    true_selection = sim_data[1][layer]
    layer_generation_counts = sim_data[3][layer]
    inferred_selection = sim_data[2][layer][0]
    best_dim_idx = np.argmax(np.abs(true_selection))

    # Find the covariance between all embeddings with the best dimension
    best_dim_values = embeddings[:, best_dim_idx]
    fixed_gen = find_fixed_gen(layer_generation_counts)
    print(f"Cutoff generation for layer {layer}: {fixed_gen}")
    total_covariances = []
    for gen in range(fixed_gen):
        # Find the covariance using a weighted approach and the population from the first selection event (gen=1)
        weights = np.average(layer_generation_counts[gen], axis=0)
        covariances = []
        for dim in range(embeddings.shape[1]):
            dim_values = embeddings[:, dim]
            mean_best = np.average(best_dim_values, weights=weights)
            mean_dim = np.average(dim_values, weights=weights)
            covariance = np.average((best_dim_values - mean_best) * (dim_values - mean_dim), weights=weights)
            covariances.append(covariance)
        covariances = np.array(covariances)
        total_covariances.append(covariances)
    avg_covariances = np.mean(total_covariances, axis=0)
    # Plot the selection coefficients in order of rank, colored by covariance with best dimension
    # Plot all three replicates
    plt.figure(figsize=(15, 5))
    plt.style.use('seaborn-v0_8-darkgrid')
    for rep in range(3):
        rep_inf_sel = inferred_selection[rep]
        
        normalized_selection = z_normalize(rep_inf_sel)
        
        # Use this:
        sorted_indices = np.argsort(normalized_selection)[::-1]  # Sort descending
        x_vals = np.arange(len(normalized_selection))  # Simple 0, 1, 2, ... for x-axis
        y_vals = normalized_selection[sorted_indices]  # Values in descending order
        covariances_sorted = avg_covariances[sorted_indices]  # Sort covariances to match
        
        plt.subplot(1, 3, rep + 1)
        scatter = plt.scatter(x_vals, y_vals, c=covariances_sorted, cmap='coolwarm', alpha=0.7)
        plt.colorbar(scatter, label='Average Covariance with Best Dimension')
        plt.title(f'Layer {layer} Replicate {rep + 1}')
        plt.xlabel('Rank of Inferred Selection Coefficient')
        plt.ylabel('Inferred Selection Coefficient (Normalized)')
        plt.axhline(0, color='black', linestyle='--')
        
        
        # For the best dimension highlight:
        best_dim_position = np.where(sorted_indices == best_dim_idx)[0]
        plt.scatter(best_dim_position, normalized_selection[best_dim_idx],
                    color='yellow', edgecolor='black', s=100, label='Best Dimension')
        
    plt.tight_layout()
    plt.show()

# plot fitness over time
def plot_fitness_over_time(sim_data):
    # For each layer, plto the fitness growth over time
    
    all_layer_fits = sim_data[0]
    all_gen_counts = sim_data[3]
    
    n_gens = len(all_gen_counts[0]) - 1
    n_reps = len(all_gen_counts[0][0])
    
    avg_fitness_over_time = {}
    for layer in range(len(all_layer_fits.keys())):
        layer_fits = all_layer_fits[layer]
        layer_counts = all_gen_counts[layer]
        avg_fitness_by_rep = []
        for gen in range(len(layer_counts)):
            gen_counts = layer_counts[gen]
            gen_fitnesses = []
            for rep in range(n_reps):
                rep_counts = gen_counts[rep]
                fitnesses = np.array(layer_fits)
                fitnesses = z_normalize(fitnesses)
                fitnesses = fitnesses / np.max(fitnesses)
                avg_fitness = np.sum(fitnesses * rep_counts) / np.sum(rep_counts)
                gen_fitnesses.append(avg_fitness)
            avg_fitness_by_rep.append(gen_fitnesses)
        # Add the replicate information to the layer
        avg_fitness_over_time[layer] = avg_fitness_by_rep

    # Plot the growtih in fitness over time for each layer and replicate
    save_every = 1
    x_vals = np.arange(0, n_gens + 1, save_every)
    
    plt.figure(figsize=(10, 6))
    plt.style.use('seaborn-v0_8-darkgrid')
    for layer in range(1, len(avg_fitness_over_time.keys())):
        layer_avg_fitness = np.array(avg_fitness_over_time[layer])
        for rep in range(n_reps):
            #print(x_vals.shape, layer_avg_fitness[:, rep].shape)
            plt.plot(x_vals, layer_avg_fitness[:, rep], label=f'Layer {layer} Replicate {rep + 1}')
    plt.xlabel('Generation')
    plt.ylabel('Average Fitness')
    plt.title('Average Fitness over Generations for Each Layer and Replicate')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
    
    
def plot_inferred_vs_true_sel(sim_data):
    # The selections of every coefficients, and the inferred selections of every coefficient across layer and generation
    detailed_selection_results = sim_data[2]
    all_sel_coeffs = sim_data[1]
    n_reps = detailed_selection_results[0][0].shape[0]
    for layer in detailed_selection_results.keys():
        layer_selection = layer
        true_selection = all_sel_coeffs[layer_selection]
        normalized_selection = z_normalize(true_selection)

        for gen in range(len(detailed_selection_results[0])):
            inferred_selection = detailed_selection_results[layer_selection][gen]
            
            fig, axs = plt.subplots(1, 3, figsize=(18, 6))
            plt.style.use('seaborn-v0_8-darkgrid')
            for rep in range(n_reps):
                x = true_selection
                y = z_normalize(inferred_selection[rep])
                
                
                axs[rep].scatter(x, y, alpha=0.5)
                axs[rep].set_title(f'Layer {layer_selection} Generation {gen * 2 + 1} Replicate {rep + 1}')
                axs[rep].set_xlabel('True Selection Coefficients')
                axs[rep].set_ylabel('Inferred Selection Coefficients')
                """axs[rep].plot([min(normalized_selection), max(normalized_selection)],
                            [min(normalized_selection), max(normalized_selection)],
                            color='red', linestyle='--')"""
            
                axs[rep].set_xlim(-0.05, 0.12)
            
            plt.tight_layout()
            plt.show()

def ordered_cov_plots(sim_data, layer=0):
    layer_df = get_df_selection(random_seed=42, selected_layer=layer,
                                    normalize_embeddings=False)[0]
    layer_df = normalize_embeddings(layer_df)
    embeddings = np.vstack(layer_df['Embedding'].tolist())

    # Get the selection coefficients
    true_selection = sim_data[1][layer]
    layer_generation_counts = sim_data[3][layer]
    inferred_selection = sim_data[2][layer][0]
    best_dim_idx = np.argmax(np.abs(true_selection))

    # Find the covariance between all embeddings with the best dimension
    best_dim_values = embeddings[:, best_dim_idx]
    
    for gen in range(50):
        print(f"Generation: {gen}")
        # Find the covariance using a weighted approach and the population from the first selection event (gen=1)
        weights = np.average(layer_generation_counts[gen], axis=0)
        covariances = []
        for dim in range(embeddings.shape[1]):
            dim_values = embeddings[:, dim]
            mean_best = np.average(best_dim_values, weights=weights)
            mean_dim = np.average(dim_values, weights=weights)
            covariance = np.average((best_dim_values - mean_best) * (dim_values - mean_dim), weights=weights)
            covariances.append(covariance)
        covariances = np.array(covariances)

        # Plot the selection coefficients in order of rank, colored by covariance with best dimension
        # Plot all three replicates
        plt.figure(figsize=(15, 5))
        plt.style.use('seaborn-v0_8-darkgrid')
        for rep in range(3):
            rep_inf_sel = inferred_selection[rep]
            
            normalized_selection = z_normalize(rep_inf_sel)
            
            # Use this:
            sorted_indices = np.argsort(normalized_selection)[::-1]  # Sort descending
            x_vals = np.arange(len(normalized_selection))  # Simple 0, 1, 2, ... for x-axis
            y_vals = normalized_selection[sorted_indices]  # Values in descending order
            covariances_sorted = covariances[sorted_indices]  # Sort covariances to match
            
            plt.subplot(1, 3, rep + 1)
            scatter = plt.scatter(x_vals, y_vals, c=covariances_sorted, cmap='coolwarm', alpha=0.7)
            plt.colorbar(scatter, label='Covariance with Best Dimension')
            plt.title(f'Layer {layer} Replicate {rep + 1}')
            plt.xlabel('Rank of Inferred Selection Coefficient')
            plt.ylabel('Inferred Selection Coefficient (Normalized)')
            plt.axhline(0, color='black', linestyle='--')
            
            
            # For the best dimension highlight:
            best_dim_position = np.where(sorted_indices == best_dim_idx)[0]
            plt.scatter(best_dim_position, normalized_selection[best_dim_idx],
                        color='yellow', edgecolor='black', s=100, label='Best Dimension')
            

        plt.tight_layout()
        plt.show()
    

from scipy.stats import rankdata, spearmanr

def comp_inf_vs_real_fits_rank(sim_data, layer, fitness='plus1'):
    """Compare the inferred and real fitness scores using ranks."""
    # For each layer, plot the fitness growth over time
    all_layer_fits = sim_data[0][layer]
    all_gen_counts = sim_data[3]
    embeddings = sim_data[4][layer]
    n_gens = len(all_gen_counts[0]) - 1
    n_reps = len(all_gen_counts[0][0])

    # Compute inferred fitness
    inferred_fits = calc_inferred_fits(sim_data, fitness=fitness,
                                       layer=layer)

    # Real fits (same across replicates)
    real_fits = np.array(all_layer_fits)

    # Rank-transform helper (average ranks for ties, 1-indexed)
    def rank_transform(x):
        return rankdata(x, method='average')

    # Rank the real fits once
    real_ranks = rank_transform(real_fits)
    n_variants = len(real_fits)

    # Rank inferred fits per replicate
    rep_ranks = {}
    for rep in range(n_reps):
        rep_ranks[rep] = rank_transform(inferred_fits[:, rep])

    # Plot — create figure with constrained_layout instead of tight_layout
    plt.close('all')
    fig, axes = plt.subplots(1, n_reps,
                             figsize=(6 * n_reps, 6))#,
                             #squeeze=False,
                             #constrained_layout=True)
    axes = axes.flatten()

    all_real_ranks = []
    all_inf_ranks = []

    for rep in range(n_reps):
        ax = axes[rep]
        r = real_ranks
        inf = rep_ranks[rep]

        ax.scatter(r, inf, alpha=0.5)
        ax.set_xlabel('Real Fitness (Rank)')
        ax.set_ylabel('Inferred Fitness (Rank)')
        ax.set_title(f'Layer {layer} — Replicate {rep}')
        ax.grid(True, alpha=0.3)

        # Diagonal reference line
        ax.plot([1, n_variants], [1, n_variants], color='red', linestyle='--')
        ax.set_xlim(0, n_variants + 1)
        ax.set_ylim(0, n_variants + 1)

        # Spearman rho
        rho, pval = spearmanr(real_fits, inferred_fits[:, rep])
        ax.annotate(f'\rho = {rho:.3f}\np = {pval:.2e}',
                     xy=(0.05, 0.95), xycoords='axes fraction',
                     ha='left', va='top',
                     fontsize=11, bbox=dict(boxstyle='round', fc='white', alpha=0.8))

        all_real_ranks.extend(r)
        all_inf_ranks.extend(inf)

    fig.suptitle(f'Real vs Inferred Fitness (Rank) — Layer {layer}',
                 fontsize=14)
    #fig.savefig(f'rank_comparison_layer_{layer}.png', dpi=100, bbox_inches='tight')
    plt.show()
    #plt.close(fig)

    # Overall Spearman correlation across all replicates
    overall_rho, overall_pval = spearmanr(all_real_ranks, all_inf_ranks)
    print(f'Overall Spearman ρ = {overall_rho:.4f}, p = {overall_pval:.2e}')
    return rep_ranks, overall_rho

  

## Simulate other methods and compare

In [ ]:
# Simulate other methods

def other_methods(df_selection, generation_counts, generation=-1):
    """Find the enrichment ratio, log ratio, and log enrichment"""
    first_gen = generation_counts[0]
    last_gen = generation_counts[generation]
    
    enrichments = []
    log_ratios = []
    #log_enrichments = []
    
    embeddings = np.vstack(df_selection['Embedding'].values)
    
    for rep in range(len(first_gen)):
        start_counts = first_gen[rep]
        end_counts = last_gen[rep]
        
        embedding_avg_before = np.sum(embeddings.T * start_counts, axis=1) / np.sum(start_counts)
        embedding_avg_after = np.sum(embeddings.T * end_counts, axis=1) / np.sum(end_counts)
        
        
        embedding_sum = np.sum(embedding_avg_before) + np.sum(embedding_avg_after) / 2
        
        enrichment = (embedding_avg_after / embedding_avg_before) / embedding_sum
        
        log_ratio = np.log((embedding_avg_after / embedding_avg_before) / embedding_sum)
        
        #log_enrichment = np.log(enrichment)
        
        log_ratios.append(log_ratio)
        enrichments.append(enrichment)
        
    return np.array(enrichments), np.array(log_ratios) #, np.array(log_enrichments)


In [ ]:
#found_sel_coeffs
enrichments, log_ratios = other_methods(df_selection, generation_counts, generation=-1)
enrichments.shape, log_ratios.shape

In [ ]:
# Plot the selection coefficients for the three replicates against each ohter
fig, axs = plt.subplots(1, 3, figsize=(18, 6))
plt.style.use('seaborn-v0_8-darkgrid')
rep_pairs = [(0, 1), (0, 2), (1, 2)]
for i, (rep1, rep2) in enumerate(rep_pairs):
    x = found_sel_coeffs[rep1]
    y = found_sel_coeffs[rep2]
    # z normalize
    #x = (x - np.mean(x)) / np.std(x)
    #y = (y - np.mean(y)) / np.std(y)
    
    r_pearson = np.corrcoef(x, y)[0, 1]
    
    print(f'Pearson correlation between Replicate {rep1 + 1} and Replicate {rep2 + 1}: {r_pearson:.3f}')
    
    axs[i].scatter(x, y, alpha=0.5)
    axs[i].set_title(f'Replicate {rep1 + 1} vs Replicate {rep2 + 1}')
    axs[i].set_xlabel(f'Selection Coefficients Replicate {rep1 + 1}')
    axs[i].set_ylabel(f'Selection Coefficients Replicate {rep2 + 1}')
    axs[i].plot([min(x), max(x)],
                [min(x), max(x)],
                color='red', linestyle='--')
    # Add the pearson scorei n top left
    axs[i].text(0.05, 0.95, f'Pearson r: {r_pearson:.3f}',
                transform=axs[i].transAxes,
                verticalalignment='top')
plt.tight_layout()

# Plot the selection coefficients for the three replicates against each ohter
fig, axs = plt.subplots(1, 3, figsize=(18, 6))
plt.style.use('seaborn-v0_8-darkgrid')
rep_pairs = [(0, 1), (0, 2), (1, 2)]
for i, (rep1, rep2) in enumerate(rep_pairs):
    x = enrichments[rep1]
    y = enrichments[rep2]
    # z normalize
    #x = (x - np.mean(x)) / np.std(x)
    #y = (y - np.mean(y)) / np.std(y)
    
    r_pearson = np.corrcoef(x, y)[0, 1]
    
    print(f'Pearson correlation between Replicate {rep1 + 1} and Replicate {rep2 + 1}: {r_pearson:.3f}')
    
    axs[i].scatter(x, y, alpha=0.5)
    axs[i].set_title(f'Replicate {rep1 + 1} vs Replicate {rep2 + 1}')
    axs[i].set_xlabel(f'enrichments  Replicate {rep1 + 1}')
    axs[i].set_ylabel(f'enrichments  Replicate {rep2 + 1}')
    axs[i].plot([min(x), max(x)],
                [min(x), max(x)],
                color='red', linestyle='--')
    # Add the pearson scorei n top left
    axs[i].text(0.05, 0.95, f'Pearson r: {r_pearson:.3f}',
                transform=axs[i].transAxes,
                verticalalignment='top')
plt.tight_layout()


# Plot the selection coefficients for the three replicates against each ohter
fig, axs = plt.subplots(1, 3, figsize=(18, 6))
plt.style.use('seaborn-v0_8-darkgrid')
rep_pairs = [(0, 1), (0, 2), (1, 2)]
for i, (rep1, rep2) in enumerate(rep_pairs):
    x = log_ratios[rep1]
    y = log_ratios[rep2]
    # z normalize
    #x = (x - np.mean(x)) / np.std(x)
    #y = (y - np.mean(y)) / np.std(y)
    
    r_pearson = np.corrcoef(x, y)[0, 1]
    
    print(f'Pearson correlation between Replicate {rep1 + 1} and Replicate {rep2 + 1}: {r_pearson:.3f}')
    
    axs[i].scatter(x, y, alpha=0.5)
    axs[i].set_title(f'Replicate {rep1 + 1} vs Replicate {rep2 + 1}')
    axs[i].set_xlabel(f'log_ratios Replicate {rep1 + 1}')
    axs[i].set_ylabel(f'log_ratios Replicate {rep2 + 1}')
    axs[i].plot([min(x), max(x)],
                [min(x), max(x)],
                color='red', linestyle='--')
    # Add the pearson scorei n top left
    axs[i].text(0.05, 0.95, f'Pearson r: {r_pearson:.3f}',
                transform=axs[i].transAxes,
                verticalalignment='top')
plt.tight_layout()







In [ ]:
# Lets run the whole simulation at every layer and calculate the average pearson r at every generation
layer_pearson_results = {}
n_layers = 31
n_gens = 10
n_reps = 3

for layer in range(n_layers):
    print(f"Running layer {layer}...")
    df_selection, initial_counts = get_df_selection(random_seed=42, selected_layer=layer)
    
    # Generate some random selection coefficients for the 640 dimensions
    np.random.seed(42)
    selection_coefficients = np.random.normal(loc=0.0, scale=0.1, size=640)
    #selection_coefficients = np.abs(selection_coefficients)  # Ensure non-negative for this example
    
    print("Running simulation...")
    generation_counts = run_simulation(selection_coefficients, initial_counts, n_gens=30)
    
    layer_results = []
    
    for gen in range(1, n_gens + 1):
        print(f"  Analyzing generation {gen}...")
        test_path = pwd + f"/simulations/layer_{layer}_gen_{gen}/"
        data = run_inference_calcs_sims(df_selection, generation_counts[:gen + 1], test_path)
        found_sel_coeffs = data[2]
        
        # Calculate pearson r for each replicate comparison
        rep_pearsons = []
        for rep1 in range(3):
            for rep2 in range(rep1 + 1, 3):
                x = found_sel_coeffs[rep1]
                y = found_sel_coeffs[rep2]
                r_pearson = np.corrcoef(x, y)[0, 1]
                rep_pearsons.append(r_pearson)
        
        
        layer_results.append(np.mean(rep_pearsons))
    
    layer_pearson_results[layer] = layer_results

In [ ]:
# Plot the layer results
plt.figure(figsize=(10, 6))
for layer in range(n_layers):
    plt.plot(range(1, n_gens + 1), layer_pearson_results[layer], label=f'Layer {layer}')
plt.xlabel('Generation')
plt.ylabel('Average Pearson r between Replicates')
plt.title('Average Pearson r between Replicates over Generations for Each Layer')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# save layer results to pickle
with open(pwd + '/layer_pearson_results.pkl', 'wb') as f:
    pickle.dump(layer_pearson_results, f)

In [ ]:
# Plot the layer results
plt.figure(figsize=(10, 6))
for layer in range(n_layers):
    plt.plot(range(1, n_gens + 1), layer_pearson_results[layer], label=f'Layer {layer}')
plt.xlabel('Generation')
plt.ylabel('Average Pearson r between Replicates')
plt.title('Average Pearson r between Replicates over Generations for Each Layer')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.ylim(0.99, 1.0)

plt.show()

## Simulations with a few really strong and the rest weak

In [ ]:
# Lets run the whole simulation at every layer and calculate the average pearson r at every generation
newsim_pearson_results = {}
n_layers = 31
n_gens = 10
n_reps = 3

for layer in range(n_layers):
    print(f"Running layer {layer}...")
    df_selection, initial_counts = get_df_selection(random_seed=42, selected_layer=layer)
    
    # Generate some random selection coefficients for the 640 dimensions
    np.random.seed(42)
    #selection_coefficients = np.random.normal(loc=0.0, scale=0.1, size=640)
    # find the range of each embedding dimension
    embedding_matrix = np.vstack(df_selection['Embedding'].values)
    embedding_ranges = embedding_matrix.max(axis=0) - embedding_matrix.min(axis=0)
    # Find the index of the highest, middle, and lowest range dimensions
    sorted_indices = np.argsort(embedding_ranges)
    high_range_idx = sorted_indices[-1]
    mid_range_idx = sorted_indices[len(sorted_indices) // 2]
    low_range_idx = sorted_indices[0]
    # Set -100 to every dimension except these three
    selection_coefficients = np.zeros(640)
    selection_coefficients[high_range_idx] = 10.0
    selection_coefficients[mid_range_idx] = 10.0
    selection_coefficients[low_range_idx] = 10.0
    
    
    print("Running simulation...")
    generation_counts = run_simulation(df_selection, selection_coefficients, 
                                       initial_counts, n_gens=30)
    
    layer_results = []
    
    for gen in range(1, n_gens + 1, 2):
        print(f"  Analyzing generation {gen}...")
        test_path = pwd + f"/simulations/layer_{layer}_gen_{gen}/"
        data = run_inference_calcs_sims(df_selection, generation_counts[:gen + 1], test_path)
        found_sel_coeffs = data[2]
        
        # Calculate pearson r for each replicate comparison
        rep_pearsons = []
        for rep1 in range(3):
            for rep2 in range(rep1 + 1, 3):
                x = found_sel_coeffs[rep1]
                y = found_sel_coeffs[rep2]
                r_pearson = np.corrcoef(x, y)[0, 1]
                rep_pearsons.append(r_pearson)
        
        layer_results.append(np.mean(rep_pearsons))
    
    newsim_pearson_results[layer] = layer_results

In [ ]:
# Plot the layer results
plt.figure(figsize=(10, 6))
for layer in range(n_layers):
    plt.plot(range(1, n_gens + 1, 2), newsim_pearson_results[layer], label=f'Layer {layer}')
plt.xlabel('Generation')
plt.ylabel('Average Pearson r between Replicates')
plt.title('Average Pearson r between Replicates over Generations for Each Layer')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
#plt.ylim(0.99, 1.0)

plt.show()

## Looking at the ranking of selection coefficients found versus real

In [ ]:
# What data do I want to save?
# The selections of every coefficients, and the inferred selections of every coefficient across layer and generation
detailed_selection_results = {}
selection_coefficients_all = {}

n_layers = 31
n_gens = 10
n_reps = 3

for layer in range(n_layers):
    print(f"Running layer {layer}...")
    df_selection, initial_counts = get_df_selection(random_seed=42, selected_layer=layer)
    
    # Generate some random selection coefficients for the 640 dimensions
    np.random.seed(42)
    #selection_coefficients = np.random.normal(loc=0.0, scale=0., size=640)
    # find the range of each embedding dimension
    np.random.seed(42)
    selection_coefficients = np.random.normal(loc=0.0, scale=1.0, size=640)
    #selection_coefficients = np.abs(selection_coefficients)  # Ensure non-negative for this example
    selection_coefficients_all[layer] = selection_coefficients
    
    print("Running simulation...")
    generation_counts = run_simulation(df_selection, selection_coefficients, 
                                       initial_counts, n_gens=30)
    layer_results = []
    
    for gen in range(1, n_gens + 1, 2):
        print(f"  Analyzing generation {gen}...")
        test_path = pwd + f"/simulations/layer_{layer}_gen_{gen}/"
        data = run_inference_calcs_sims(df_selection, generation_counts[:gen + 1], test_path)
        found_sel_coeffs = data[2]
        layer_results.append(found_sel_coeffs)
    
    detailed_selection_results[layer] = layer_results

In [ ]:
# Now save the detailed selection results and selection coefficients all to pickle
with open(pwd + '/detailed_selection_results.pkl', 'wb') as f:
    pickle.dump(detailed_selection_results, f)
with open(pwd + '/selection_coefficients_all.pkl', 'wb') as f:
    pickle.dump(selection_coefficients_all, f)

In [ ]:
len(detailed_selection_results[0])
n_gens

In [ ]:
# Now compare the ranks of the true vs inferred selection coefficients for each layer and generation
from scipy.stats import rankdata
detailed_rank_results = {}
for layer in range(n_layers):
    layer_rank_results = []
    true_selection = selection_coefficients_all[layer]
    for gen in range(0, len(detailed_selection_results[0])):
        inferred_selection = detailed_selection_results[layer][gen]
        rep_rank_pearsons = []
        for rep in range(n_reps):
            true_ranks = rankdata(true_selection)
            inferred_ranks = rankdata(inferred_selection[rep])
            r_pearson = np.corrcoef(true_ranks, inferred_ranks)[0, 1]
            rep_rank_pearsons.append(r_pearson)
        layer_rank_results.append(np.mean(rep_rank_pearsons))
    detailed_rank_results[layer] = layer_rank_results
    
# Plot the layer results
plt.figure(figsize=(10, 6))
for layer in range(n_layers):
    plt.plot(range(1, n_gens + 1, 2), detailed_rank_results[layer], label=f'Layer {layer}')
plt.xlabel('Generation')
plt.ylabel('Average Rank Pearson r between True and Inferred')
plt.title('Average Rank Pearson r between True and Inferred Selection Coefficients over Generations for Each Layer')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# The selections of every coefficients, and the inferred selections of every coefficient across layer and generation
layer_selection = 12
true_selection = selection_coefficients_all[layer_selection]
true_ranks = rankdata(true_selection)

for gen in range(len(detailed_selection_results[0])):
    inferred_selection = detailed_selection_results[layer_selection][gen]
    
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    plt.style.use('seaborn-v0_8-darkgrid')
    for rep in range(n_reps):
        x = true_ranks
        y = rankdata(inferred_selection[rep])
        
        
        axs[rep].scatter(x, y, alpha=0.5)
        axs[rep].set_title(f'Layer {layer_selection} Generation {gen * 2 + 1} Replicate {rep + 1}')
        axs[rep].set_xlabel('True Selection Coefficients')
        axs[rep].set_ylabel('Inferred Selection Coefficients')
        axs[rep].plot([min(true_ranks), max(true_ranks)],
                      [min(true_ranks), max(true_ranks)],
                      color='red', linestyle='--')
    plt.tight_layout()
    plt.show()


In [ ]:
# The selections of every coefficients, and the inferred selections of every coefficient across layer and generation
layer_selection = 12
true_selection = selection_coefficients_all[layer_selection]
normalized_selection = z_normalize(true_selection)

for gen in range(len(detailed_selection_results[0])):
    inferred_selection = detailed_selection_results[layer_selection][gen]
    
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    plt.style.use('seaborn-v0_8-darkgrid')
    for rep in range(n_reps):
        x = true_selection
        y = z_normalize(inferred_selection[rep])
        
        
        axs[rep].scatter(x, y, alpha=0.5)
        axs[rep].set_title(f'Layer {layer_selection} Generation {gen * 2 + 1} Replicate {rep + 1}')
        axs[rep].set_xlabel('True Selection Coefficients')
        axs[rep].set_ylabel('Inferred Selection Coefficients')
        axs[rep].plot([min(normalized_selection), max(normalized_selection)],
                      [min(normalized_selection), max(normalized_selection)],
                      color='red', linestyle='--')
    plt.tight_layout()
    plt.show()


## Investigating this fucking data some more.

Q's to answer

1. What does the range of fitness scores look like?

2. How does a range of selection coefficients map to a range of fitness scores for each layer based on different methods?

3. How does a range of fitness scores map to a rank-rank correlation of true and inferred selection coefficients?

4. Is there a bug in my inference code?
    * Check with one SUPER strong selection coefficient and the rest weak to see if it picks it out
    * Try this with the largest, middle, and smallest range of embeddings to see if it is simply impossible to infer under a given size

5. What is the minimum range of selection coefficients that can be inferred with a given embedding size?

 


### 1. What does the range of fitness scores look like?

In [ ]:
# What data do I want to save?
# The selections of every coefficients, and the inferred selections of every coefficient across layer and generation
detailed_selection_results = {}
selection_coefficients_all = {}
all_layer_fits = {}

n_layers = 31
n_gens = 50
n_reps = 3

for layer in range(n_layers):
    print(f"Running layer {layer}...")
    df_selection, initial_counts = get_df_selection(random_seed=42, selected_layer=layer)
    
    # Generate some random selection coefficients for the 640 dimensions
    np.random.seed(42)
    selection_coefficients = np.random.normal(loc=0.0, scale=1.0, size=640)
    selection_coefficients_all[layer] = selection_coefficients
    
    print("Running simulation...")
    generation_counts, layer_fits = run_simulation(df_selection, selection_coefficients, 
                                       initial_counts, n_gens=n_gens)
    
    all_layer_fits[layer] = layer_fits
    
    layer_results = []
    
    for gen in range(1, n_gens + 1, 5):
        print(f"  Analyzing generation {gen}...")
        test_path = pwd + f"/simulations/layer_{layer}_gen_{gen}/"
        data = run_inference_calcs_sims(df_selection, generation_counts[:gen + 1], test_path)
        found_sel_coeffs = data[2]
        layer_results.append(found_sel_coeffs)
    
    detailed_selection_results[layer] = layer_results

In [ ]:
# Analyze the spread of fitness scores for each layer, range, median, std_dev

for layer in range(len(all_layer_fits.keys())):
    layer_fits = all_layer_fits[layer]
    print(f"Layer {layer} Fitness Statistics:")
    print(f"  Mean: {np.mean(layer_fits)}")
    print(f"  Median: {np.median(layer_fits)}")
    print(f"  Std Dev: {np.std(layer_fits)}")
    print(f"  Min: {np.min(layer_fits)}")
    print(f"  Max: {np.max(layer_fits)}")

In [ ]:
# Analyze the range of selection coefficients to the range of fitnesses
selection_ranges = {}
for layer in range(len(selection_coefficients_all.keys())):
    selection_coeffs = selection_coefficients_all[layer]
    layer_fits = all_layer_fits[layer]
    selection_range = np.max(selection_coeffs) - np.min(selection_coeffs)
    fitness_range = np.max(layer_fits) - np.min(layer_fits)
    selection_ranges[layer] = (selection_range, fitness_range)
    print(f"Layer {layer}: Selection Coefficient Range = {selection_range}, Fitness Range = {fitness_range}")

In [ ]:
# The selections of every coefficients, and the inferred selections of every coefficient across layer and generation
from scipy.stats import rankdata
layer_selection = 15
true_selection = selection_coefficients_all[layer_selection]
true_ranks = rankdata(true_selection)

for gen in range(len(detailed_selection_results[0])):
    inferred_selection = detailed_selection_results[layer_selection][gen]
    
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    plt.style.use('seaborn-v0_8-darkgrid')
    for rep in range(n_reps):
        x = true_ranks
        y = rankdata(inferred_selection[rep])
        
        
        axs[rep].scatter(x, y, alpha=0.5)
        axs[rep].set_title(f'Layer {layer_selection} Generation {gen * 2 + 1} Replicate {rep + 1}')
        axs[rep].set_xlabel('True Selection Coefficients')
        axs[rep].set_ylabel('Inferred Selection Coefficients')
        axs[rep].plot([min(true_ranks), max(true_ranks)],
                      [min(true_ranks), max(true_ranks)],
                      color='red', linestyle='--')
    plt.tight_layout()
    plt.show()


### Alright, this range of fitness scores is way too low, let's try a few really high ones

In [ ]:
# What data do I want to save?
# The selections of every coefficients, and the inferred selections of every coefficient across layer and generation
new_all_layer_fits = {}
new_all_selection_coefficients = {}
new_detailed_selection_results = {}
all_generation_counts = {}

n_layers = 31
n_gens = 200
n_reps = 3

for layer in range(n_layers):
    print(f"Running layer {layer}...")
    df_selection, initial_counts = get_df_selection(random_seed=42, selected_layer=layer)
    # Generate some random selection coefficients for the 640 dimensions
    np.random.seed(42)
    selection_coefficients = np.random.normal(loc=0.0, scale=1.0, size=640)
    
    # Find the range of embeddings for each dimension
    embedding_matrix = np.vstack(df_selection['Embedding'].tolist())
    embedding_ranges = embedding_matrix.max(axis=0) - embedding_matrix.min(axis=0)
    # Find the indices of the top, lowest, and middle range dimensions
    sorted_indices = np.argsort(embedding_ranges)
    high_range_idx = sorted_indices[-1]
    mid_range_idx = sorted_indices[len(sorted_indices) // 2]
    low_range_idx = sorted_indices[0]
    # Give these dimensions higher selection coefficients
    selection_coefficients[high_range_idx] = 100.0
    selection_coefficients[mid_range_idx] = 99.0
    selection_coefficients[low_range_idx] = 98.0
    
    
    new_all_selection_coefficients[layer] = selection_coefficients
    
    print("Running simulation...")
    generation_counts, layer_fits = run_simulation(df_selection, selection_coefficients, 
                                       initial_counts, n_gens=n_gens)
    all_generation_counts[layer] = generation_counts
    
    print(f"shape of generation counts for this layer: {np.array(generation_counts).shape}")
    new_all_layer_fits[layer] = layer_fits
    
    layer_results = []
    
    gen=10
    print(f"  Analyzing generation {gen}...")
    test_path = pwd + f"/simulations/layer_{layer}_gen_{gen}/"
    data = run_inference_calcs_sims(df_selection, generation_counts[:gen + 1], test_path)
    found_sel_coeffs = data[2]
    layer_results.append(found_sel_coeffs)
    
    new_detailed_selection_results[layer] = layer_results

In [ ]:
# The selections of every coefficients, and the inferred selections of every coefficient across layer and generation
layer_selection = 12
for layer in new_detailed_selection_results.keys():
    layer_selection = layer
    true_selection = new_all_selection_coefficients[layer_selection]
    #normalized_selection = z_normalize(true_selection)

    for gen in range(len(new_detailed_selection_results[0])):
        inferred_selection = new_detailed_selection_results[layer_selection][gen]
        
        fig, axs = plt.subplots(1, 3, figsize=(18, 6))
        plt.style.use('seaborn-v0_8-darkgrid')
        for rep in range(n_reps):
            x = true_selection
            y = z_normalize(inferred_selection[rep])
            
            
            axs[rep].scatter(x, y, alpha=0.5)
            axs[rep].set_title(f'Layer {layer_selection} Generation {gen * 2 + 1} Replicate {rep + 1}')
            axs[rep].set_xlabel('True Selection Coefficients')
            axs[rep].set_ylabel('Inferred Selection Coefficients')
            axs[rep].plot([min(normalized_selection), max(normalized_selection)],
                        [min(normalized_selection), max(normalized_selection)],
                        color='red', linestyle='--')
        plt.tight_layout()
        plt.show()


In [ ]:
# Analyze the spread of fitness scores for each layer, range, median, std_dev

for layer in range(len(new_all_layer_fits.keys())):
    layer_fits = new_all_layer_fits[layer]
    print(f"Layer {layer} Fitness Statistics:")
    print(f"  Mean: {np.mean(layer_fits)}")
    print(f"  Median: {np.median(layer_fits)}")
    print(f"  Std Dev: {np.std(layer_fits)}")
    print(f"  Min: {np.min(layer_fits)}")
    print(f"  Max: {np.max(layer_fits)}")

# Fitness over time

In [ ]:
# What data do I want to save?
# The selections of every coefficients, and the inferred selections of every coefficient across layer and generation
fit_all_layer_fits = {}
fit_generation_counts = {}
fit_all_selection_coefficients = {}

n_layers = 31
n_gens = 200
n_reps = 3

for layer in range(n_layers):
    print(f"Running layer {layer}...")
    df_selection, initial_counts = get_df_selection(random_seed=42, selected_layer=layer)
    # Generate some random selection coefficients for the 640 dimensions
    np.random.seed(42)
    selection_coefficients = np.random.normal(loc=0.0, scale=0.0, size=640)
    
    # Find the range of embeddings for each dimension
    embedding_matrix = np.vstack(df_selection['Embedding'].tolist())
    embedding_ranges = embedding_matrix.max(axis=0) - embedding_matrix.min(axis=0)
    # Find the indices of the top, lowest, and middle range dimensions
    sorted_indices = np.argsort(embedding_ranges)
    high_range_idx = sorted_indices[-1]
    mid_range_idx = sorted_indices[len(sorted_indices) // 2]
    low_range_idx = sorted_indices[0]
    # Give these dimensions higher selection coefficients
    selection_coefficients[high_range_idx] = 100.0
    selection_coefficients[mid_range_idx] = 99.0
    selection_coefficients[low_range_idx] = 98.0 


    
    
    fit_all_selection_coefficients[layer] = selection_coefficients
    
    print("Running simulation...")
    generation_counts, layer_fits = run_simulation(df_selection, selection_coefficients, 
                                       initial_counts, n_gens=n_gens)
    fit_generation_counts[layer] = generation_counts
    
    print(f"shape of generation counts for this layer: {np.array(generation_counts).shape}")
    fit_all_layer_fits[layer] = layer_fits


In [ ]:
# For each layer, plto the fitness growth over time
avg_fitness_over_time = {}
for layer in range(len(fit_all_layer_fits.keys())):
    layer_fits = fit_all_layer_fits[layer]
    layer_counts = fit_generation_counts[layer]
    avg_fitness_by_rep = []
    for gen in range(len(layer_counts)):
        gen_counts = layer_counts[gen]
        gen_fitnesses = []
        for rep in range(n_reps):
            rep_counts = gen_counts[rep]
            fitnesses = np.array(layer_fits)
            fitnesses = z_normalize(fitnesses)
            fitnesses = fitnesses / np.max(fitnesses)
            avg_fitness = np.sum(fitnesses * rep_counts) / np.sum(rep_counts)
            gen_fitnesses.append(avg_fitness)
        avg_fitness_by_rep.append(gen_fitnesses)
    # Add the replicate information to the layer
    avg_fitness_over_time[layer] = avg_fitness_by_rep

# Plot the growtih in fitness over time for each layer and replicate
plt.figure(figsize=(10, 6))
plt.style.use('seaborn-v0_8-darkgrid')
for layer in range(len(avg_fitness_over_time.keys())):
    layer_avg_fitness = np.array(avg_fitness_over_time[layer])
    for rep in range(n_reps):
        plt.plot(range(len(layer_avg_fitness)), layer_avg_fitness[:, rep], label=f'Layer {layer} Replicate {rep + 1}')
plt.xlabel('Generation')
plt.ylabel('Average Fitness')
plt.title('Average Fitness over Generations for Each Layer and Replicate')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
# make x a log scale
plt.xscale('log')
plt.show()

In [ ]:
embedding_matrix = np.vstack(df_selection['Embedding'].tolist())

In [ ]:
print(embedding_matrix[0].shape)
print(embedding_matrix[0].max(), embedding_matrix[0].min())
print(np.mean(embedding_matrix[0]), np.std(embedding_matrix[0]))

## CLIP THE EMBEDDINGS AND MAKE MULTIPLICATIVE FITNESS AGAIN

In [ ]:
# Plot the embedding values for the first individual
plt.figure(figsize=(10, 6))
plt.style.use('seaborn-v0_8-darkgrid')
plt.plot(range(640), embedding_matrix[0], label='Embedding Values')
plt.xlabel('Embedding Dimension')
plt.ylabel('Embedding Value')


In [ ]:
# What data do I want to save?
# The selections of every coefficients, and the inferred selections of every coefficient across layer and generation
updated_all_layer_fits = {}
updated_generation_counts = {}

n_layers = 31
n_gens = 10000
n_reps = 3

for layer in range(n_layers):
    print(f"Running layer {layer}...")
    df_selection, initial_counts = get_df_selection(random_seed=42, selected_layer=layer)
    # Generate some random selection coefficients for the 640 dimensions
    np.random.seed(42)
    selection_coefficients = np.random.normal(loc=0.0, scale=0.0, size=640)
    
    # Find the range of embeddings for each dimension
    embedding_matrix = np.vstack(df_selection['Embedding'].tolist())
    embedding_ranges = embedding_matrix.max(axis=0) - embedding_matrix.min(axis=0)
    # Find the indices of the top, lowest, and middle range dimensions
    sorted_indices = np.argsort(embedding_ranges)
    high_range_idx = sorted_indices[-1]
    mid_range_idx = sorted_indices[len(sorted_indices) // 2]
    low_range_idx = sorted_indices[0]
    # Give these dimensions higher selection coefficients
    selection_coefficients[high_range_idx] = 0.10 
    selection_coefficients[mid_range_idx] = 0.08
    selection_coefficients[low_range_idx] = 0.05 
    
    print("Running simulation...")
    generation_counts, layer_fits = run_simulation(df_selection, selection_coefficients, 
                                       initial_counts, n_gens=n_gens, embedding_clip=(-1,1),
                                       save_every=100)
    updated_generation_counts[layer] = generation_counts
    updated_all_layer_fits[layer] = layer_fits


In [ ]:
# Analyze the spread of fitness scores for each layer, range, median, std_dev

for layer in range(len(updated_all_layer_fits.keys())):
    layer_fits = updated_all_layer_fits[layer]
    print(f"Layer {layer} Fitness Statistics:")
    print(f"  Mean: {np.mean(layer_fits)}")
    print(f"  Median: {np.median(layer_fits)}")
    print(f"  Std Dev: {np.std(layer_fits)}")
    print(f"  Min: {np.min(layer_fits)}")
    print(f"  Max: {np.max(layer_fits)}")

In [ ]:
# For each layer, plto the fitness growth over time
avg_fitness_over_time = {}
for layer in range(len(updated_all_layer_fits.keys())):
    layer_fits = updated_all_layer_fits[layer]
    layer_counts = updated_generation_counts[layer]
    avg_fitness_by_rep = []
    for gen in range(len(layer_counts)):
        gen_counts = layer_counts[gen]
        gen_fitnesses = []
        for rep in range(n_reps):
            rep_counts = gen_counts[rep]
            fitnesses = np.array(layer_fits)
            fitnesses = z_normalize(fitnesses)
            fitnesses = fitnesses / np.max(fitnesses)
            avg_fitness = np.sum(fitnesses * rep_counts) / np.sum(rep_counts)
            gen_fitnesses.append(avg_fitness)
        avg_fitness_by_rep.append(gen_fitnesses)
    # Add the replicate information to the layer
    avg_fitness_over_time[layer] = avg_fitness_by_rep

# Plot the growtih in fitness over time for each layer and replicate
save_every = 100
x_vals = np.arange(0, n_gens + 1, save_every)
plt.figure(figsize=(10, 6))
plt.style.use('seaborn-v0_8-darkgrid')
for layer in range(1, len(avg_fitness_over_time.keys())):
    layer_avg_fitness = np.array(avg_fitness_over_time[layer])
    for rep in range(n_reps):
        plt.plot(x_vals, layer_avg_fitness[:, rep], label=f'Layer {layer} Replicate {rep + 1}')
plt.xlabel('Generation')
plt.ylabel('Average Fitness')
plt.title('Average Fitness over Generations for Each Layer and Replicate')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
# make x a log scale
#plt.xscale('log')
plt.show()

Okay, looks better. Let's see how this effects the selection coefficients again 

## Rerunning the simulation code with new normalizations

In [ ]:
# What data do I want to save?
# The selections of every coefficients, and the inferred selections of every coefficient across layer and generation
new_all_layer_fits = {}
new_all_selection_coefficients = {}
new_detailed_selection_results = {}
all_generation_counts = {}

n_layers = 31
n_gens = 3000
n_reps = 3

for layer in range(n_layers):
    print(f"Running layer {layer}...")
    df_selection, initial_counts = get_df_selection(random_seed=42, selected_layer=layer)
    # Generate some random selection coefficients for the 640 dimensions
    np.random.seed(42)
    selection_coefficients = np.random.normal(loc=0.0, scale=0.0, size=640)
    
    # Find the range of embeddings for each dimension
    embedding_matrix = np.vstack(df_selection['Embedding'].values)
    embedding_ranges = embedding_matrix.max(axis=0) - embedding_matrix.min(axis=0)
    # Find the indices of the top, lowest, and middle range dimensions
    sorted_indices = np.argsort(embedding_ranges)
    high_range_idx = sorted_indices[-1]
    mid_range_idx = sorted_indices[len(sorted_indices) // 2]
    low_range_idx = sorted_indices[0]
    # Give these dimensions higher selection coefficients
    selection_coefficients[high_range_idx] = 0.10
    selection_coefficients[mid_range_idx] = 0.08
    selection_coefficients[low_range_idx] = 0.05
    
    
    new_all_selection_coefficients[layer] = selection_coefficients
    
    print("Running simulation...")
    generation_counts, layer_fits = run_simulation(df_selection, selection_coefficients, 
                                       initial_counts, n_gens=n_gens, embedding_clip=(-1,1),
                                        save_every=10)
    all_generation_counts[layer] = generation_counts
    
    print(f"shape of generation counts for this layer: {np.array(generation_counts).shape}")
    new_all_layer_fits[layer] = layer_fits
    
    layer_results = []
    
    gen=3000
    print(f"  Analyzing generation {gen}...")
    test_path = pwd + f"/simulations/layer_{layer}_gen_{gen}/"
    data = run_inference_calcs_sims(df_selection, generation_counts[:gen + 1], test_path)
    found_sel_coeffs = data[2]
    layer_results.append(found_sel_coeffs)
    
    new_detailed_selection_results[layer] = layer_results

In [ ]:
# The selections of every coefficients, and the inferred selections of every coefficient across layer and generation
for layer in new_detailed_selection_results.keys():
    layer_selection = layer
    true_selection = new_all_selection_coefficients[layer_selection]
    normalized_selection = z_normalize(true_selection)

    for gen in range(len(new_detailed_selection_results[0])):
        inferred_selection = new_detailed_selection_results[layer_selection][gen]
        
        fig, axs = plt.subplots(1, 3, figsize=(18, 6))
        plt.style.use('seaborn-v0_8-darkgrid')
        for rep in range(n_reps):
            x = true_selection
            y = z_normalize(inferred_selection[rep])
            
            
            axs[rep].scatter(x, y, alpha=0.5)
            axs[rep].set_title(f'Layer {layer_selection} Generation {gen * 2 + 1} Replicate {rep + 1}')
            axs[rep].set_xlabel('True Selection Coefficients')
            axs[rep].set_ylabel('Inferred Selection Coefficients')
            """axs[rep].plot([min(normalized_selection), max(normalized_selection)],
                        [min(normalized_selection), max(normalized_selection)],
                        color='red', linestyle='--')"""
        
        plt.xlim(min(normalized_selection), max(normalized_selection))
        
        plt.tight_layout()
        plt.show()


# Noramlizing embeddings across the other axis

In [ ]:
def normalize_embeddings(df):
    #print(df.head())
    embedding_array = np.array([x for x in df["Embedding"].to_list()])
    #print(embedding_array.shape)
    z_embeddings = np.zeros_like(embedding_array)
    dimensions = embedding_array.shape[1]
    for dim in range(dimensions):
        z_embeddings[:, dim] = z_normalize(embedding_array[:, dim])
        
    # insert back into the dataframe
    df["Embedding"] = [z_embeddings[i] for i in range(z_embeddings.shape[0])]
    return df

### Running just the fitness part of the simulation

In [ ]:
# What data do I want to save?
# The selections of every coefficients, and the inferred selections of every coefficient across layer and generation
updated_all_layer_fits = {}
updated_generation_counts = {}

n_layers = 31
n_gens = 100
n_reps = 3

for layer in range(n_layers):
    print(f"Running layer {layer}...")
    df_selection, initial_counts = get_df_selection(random_seed=42, selected_layer=layer,
                                                    normalize_embeddings=False)
    
    df_selection = normalize_embeddings(df_selection)
    
    # Generate some random selection coefficients for the 640 dimensions
    np.random.seed(42)
    selection_coefficients = np.random.normal(loc=0.0, scale=0.0, size=640)
    
    # Find the range of embeddings for each dimension
    embedding_matrix = np.vstack(df_selection['Embedding'].values)
    embedding_ranges = embedding_matrix.max(axis=0) - embedding_matrix.min(axis=0)
    # Find the indices of the top, lowest, and middle range dimensions
    sorted_indices = np.argsort(embedding_ranges)
    high_range_idx = sorted_indices[-1]
    mid_range_idx = sorted_indices[len(sorted_indices) // 2]
    low_range_idx = sorted_indices[0]
    # Give these dimensions higher selection coefficients
    selection_coefficients[high_range_idx] = 0.10
    selection_coefficients[mid_range_idx] = 0.08
    selection_coefficients[low_range_idx] = 0.05
    
    
    print("Running simulation...")
    generation_counts, layer_fits = run_simulation(df_selection, selection_coefficients, 
                                       initial_counts, n_gens=n_gens, embedding_clip=None,
                                       save_every=1)
    updated_generation_counts[layer] = generation_counts
    updated_all_layer_fits[layer] = layer_fits


### Plotting results

In [ ]:
# For each layer, plto the fitness growth over time
avg_fitness_over_time = {}
for layer in range(len(updated_all_layer_fits.keys())):
    layer_fits = updated_all_layer_fits[layer]
    layer_counts = updated_generation_counts[layer]
    avg_fitness_by_rep = []
    for gen in range(len(layer_counts)):
        gen_counts = layer_counts[gen]
        gen_fitnesses = []
        for rep in range(n_reps):
            rep_counts = gen_counts[rep]
            fitnesses = np.array(layer_fits)
            fitnesses = z_normalize(fitnesses)
            fitnesses = fitnesses / np.max(fitnesses)
            avg_fitness = np.sum(fitnesses * rep_counts) / np.sum(rep_counts)
            gen_fitnesses.append(avg_fitness)
        avg_fitness_by_rep.append(gen_fitnesses)
    # Add the replicate information to the layer
    avg_fitness_over_time[layer] = avg_fitness_by_rep

# Plot the growtih in fitness over time for each layer and replicate
save_every = 1
x_vals = np.arange(0, n_gens + 1, save_every)
plt.figure(figsize=(10, 6))
plt.style.use('seaborn-v0_8-darkgrid')
for layer in range(1, len(avg_fitness_over_time.keys())):
    layer_avg_fitness = np.array(avg_fitness_over_time[layer])
    for rep in range(n_reps):
        plt.plot(x_vals, layer_avg_fitness[:, rep], label=f'Layer {layer} Replicate {rep + 1}')
plt.xlabel('Generation')
plt.ylabel('Average Fitness')
plt.title('Average Fitness over Generations for Each Layer and Replicate')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
# make x a log scale
#plt.xscale('log')
plt.show()

### Running the simulation again

In [ ]:
# What data do I want to save?
# The selections of every coefficients, and the inferred selections of every coefficient across layer and generation
new_all_layer_fits = {}
new_all_selection_coefficients = {}
new_detailed_selection_results = {}
all_generation_counts = {}

n_layers = 31
n_gens = 50
n_reps = 3

high_indices = []
mid_indicies = []
low_indicies = []

for layer in range(n_layers):
    print(f"Running layer {layer}...")
    df_selection, initial_counts = get_df_selection(random_seed=42, selected_layer=layer,
                                                    normalize_embeddings=False)
    
    df_selection = normalize_embeddings(df_selection)
    
    # Generate some random selection coefficients for the 640 dimensions
    np.random.seed(42)
    selection_coefficients = np.random.normal(loc=0.0, scale=0.0, size=640)
    
    # Find the range of embeddings for each dimension
    embedding_matrix = np.vstack(df_selection['Embedding'].values)
    embedding_ranges = embedding_matrix.max(axis=0) - embedding_matrix.min(axis=0)
    # Find the indices of the top, lowest, and middle range dimensions
    sorted_indices = np.argsort(embedding_ranges)
    high_range_idx = sorted_indices[-1]
    mid_range_idx = sorted_indices[len(sorted_indices) // 2]
    low_range_idx = sorted_indices[0]
    # Give these dimensions higher selection coefficients
    selection_coefficients[high_range_idx] = 0.10
    selection_coefficients[mid_range_idx] = 0.08
    selection_coefficients[low_range_idx] = 0.05
    
    
    new_all_selection_coefficients[layer] = selection_coefficients
    
    print("Running simulation...")
    generation_counts, layer_fits = run_simulation(df_selection, selection_coefficients, 
                                       initial_counts, n_gens=n_gens, embedding_clip=(-1,1),
                                        save_every=1)
    all_generation_counts[layer] = generation_counts
    
    print(f"shape of generation counts for this layer: {np.array(generation_counts).shape}")
    new_all_layer_fits[layer] = layer_fits
    
    layer_results = []
    
    gen=50
    print(f"  Analyzing generation {gen}...")
    test_path = pwd + f"/simulations/layer_{layer}_gen_{gen}/"
    data = run_inference_calcs_sims(df_selection, generation_counts[:gen + 1], test_path)
    found_sel_coeffs = data[2]
    layer_results.append(found_sel_coeffs)
    
    new_detailed_selection_results[layer] = layer_results

In [ ]:
# The selections of every coefficients, and the inferred selections of every coefficient across layer and generation
for layer in new_detailed_selection_results.keys():
    layer_selection = layer
    true_selection = new_all_selection_coefficients[layer_selection]
    normalized_selection = z_normalize(true_selection)

    for gen in range(len(new_detailed_selection_results[0])):
        inferred_selection = new_detailed_selection_results[layer_selection][gen]
        
        fig, axs = plt.subplots(1, 3, figsize=(18, 6))
        plt.style.use('seaborn-v0_8-darkgrid')
        for rep in range(n_reps):
            x = true_selection
            y = z_normalize(inferred_selection[rep])
            
            
            axs[rep].scatter(x, y, alpha=0.5)
            axs[rep].set_title(f'Layer {layer_selection} Generation {gen * 2 + 1} Replicate {rep + 1}')
            axs[rep].set_xlabel('True Selection Coefficients')
            axs[rep].set_ylabel('Inferred Selection Coefficients')
            """axs[rep].plot([min(normalized_selection), max(normalized_selection)],
                        [min(normalized_selection), max(normalized_selection)],
                        color='red', linestyle='--')"""
        
        plt.xlim(-0.05, 0.12)
        
        plt.tight_layout()
        plt.show()
        
    # Get the average change in embeddings over time too
    #embedding_matrix = np.vstack(df_selection['Embedding'].values)
    #start_freq = all_generation_counts[layer][0]
 
    

In [ ]:
# Getting the average change in embedding over time too

new_detailed_selection_results
all_generation_counts

avg_embeddings = np.zeros((n_layers, n_gens, 640))

chosen_replicate = 0

for layer in range(n_layers):
    df_selection, initial_counts = get_df_selection(random_seed=42, selected_layer=layer,
                                                    normalize_embeddings=False)
    
    df_selection = normalize_embeddings(df_selection)
    
    generation_counts = all_generation_counts[layer]
    
    embedding_matrix = np.vstack(df_selection['Embedding'].values)
    
    for gen in range(len(generation_counts)):
        gen_counts = generation_counts[gen]
        avg_embedding = np.zeros(640)
        total_count = 0
        
        # Calculate the weighted average embedding for this generation
        rep_counts = gen_counts[chosen_replicate]
        for i in range(len(rep_counts)):
            count = rep_counts[i]
            embedding = embedding_matrix[i]
            avg_embedding += embedding * count
            total_count += count
        avg_embedding /= total_count
        avg_embeddings[layer, gen, :] = avg_embedding
        



In [ ]:
# Now Plot the Change in Embedding against the selection coefficients
for layer in range(n_layers):
    inferred_selection = new_detailed_selection_results[layer][-1][chosen_replicate]
    normalized_selection = z_normalize(inferred_selection)
    true_selection = new_all_selection_coefficients[layer]
    
    
    total_change = avg_embeddings[layer, -1, :] - avg_embeddings[layer, 0, :]
    
    
    
    print(f"Normalized_selection Change shape: {normalized_selection.shape}, Total_change shape: {total_change.shape}")
    # Plot the change in embedding vs selection coefficient
    plt.figure(figsize=(8, 6))
    plt.style.use('seaborn-v0_8-darkgrid')
    
    # Color by the true selection
    
    
    plt.scatter(normalized_selection, total_change, alpha=0.5,
                c=true_selection, cmap='viridis')
    plt.title(f'Layer {layer} Change in Embedding vs Selection Coefficient')
    plt.xlabel('Normalized Selection Coefficient')
    plt.ylabel('Change in Embedding Value')
    plt.axhline(0, color='red', linestyle='--')
    plt.axvline(0, color='red', linestyle='--')
    plt.tight_layout()
    plt.show()
    

## Investigating the effect of covariance on the hitchhiking of selection coefficients

In [ ]:
"""def generate_selection(embeddings):
    embedding_ranges = embeddings.max(axis=0) - embeddings.min(axis=0)
    # Find the indices of the top, lowest, and middle range dimensions
    sorted_indices = np.argsort(embedding_ranges)
    high_range_idx = sorted_indices[-1]
    # Give these dimensions higher selection coefficients
    selection_coefficients[high_range_idx] = 0.10
    return selection_coefficients

def get_simulation_results(n_layers, n_gens, n_reps, 
                           sel_func=generate_selection,
                           inference=True, fitness='plus1',
                           save_every=1):
    all_layer_fits = {}
    all_selection_coefficients = {}
    detailed_selection_results = {}
    all_generation_counts = {}

    for layer in range(n_layers):
        print(f"Running layer {layer}...")
        df_selection, initial_counts = get_df_selection(random_seed=42, selected_layer=layer,
                                                        normalize_embeddings=False)
        df_selection = normalize_embeddings(df_selection)

        # Generate selection coefficients using the provided function
        embedding_matrix = np.vstack(df_selection['Embedding'].values)
        selection_coefficients = sel_func(embedding_matrix)
        all_selection_coefficients[layer] = selection_coefficients
        
        print("Running simulation...")
        generation_counts, layer_fits = run_simulation(df_selection, selection_coefficients, 
                                           initial_counts, n_gens=n_gens, save_every=save_every,
                                           fitness=fitness)
        all_generation_counts[layer] = generation_counts
        all_layer_fits[layer] = layer_fits

        if inference:
            gen=n_gens
            print(f"  Analyzing generation {gen}...")
            layer_results = []
            test_path = pwd + f"/simulations/layer_{layer}_gen_{gen}/"
            data = run_inference_calcs_sims(df_selection, generation_counts[:gen + 1], test_path)
            found_sel_coeffs = data[2]
            layer_results.append(found_sel_coeffs)

            detailed_selection_results[layer] = layer_results

    return all_layer_fits, all_selection_coefficients, detailed_selection_results, all_generation_counts, embedding_matrix
    
    
"""

In [ ]:
sim_results = get_simulation_results(n_layers=31, n_gens=50, n_reps=3,
                                        sel_func=generate_selection)

In [ ]:
import numpy as np
from scipy.stats import rankdata
import matplotlib.pyplot as plt


for layer in sim_results[2].keys():
    layer_df = get_df_selection(random_seed=42, selected_layer=layer,
                                 normalize_embeddings=False)[0]
    layer_df = normalize_embeddings(layer_df)
    embeddings = np.vstack(layer_df['Embedding'].values)
    
    # Get the selection coefficients
    true_selection = sim_results[1][layer]
    best_dim_idx = np.argmax(np.abs(true_selection))
    
    # Find the covariance between all embeddings with the best dimension
    best_dim_values = embeddings[:, best_dim_idx]
    covariances = []
    for dim in range(embeddings.shape[1]):
        dim_values = embeddings[:, dim]
        covariance = np.cov(best_dim_values, dim_values)[0, 1]
        covariances.append(covariance)
    covariances = np.array(covariances)
    
    # Plot the selection coefficients in order of rank, colored by covariance with best dimension
    # Plot all three replicates
    plt.figure(figsize=(15, 5))
    plt.style.use('seaborn-v0_8-darkgrid')
    for rep in range(3):
        inferred_selection = sim_results[2][layer][0][rep]
        
        normalized_selection = z_normalize(inferred_selection)
        
        # Use this:
        sorted_indices = np.argsort(normalized_selection)[::-1]  # Sort descending
        x_vals = np.arange(len(normalized_selection))  # Simple 0, 1, 2, ... for x-axis
        y_vals = normalized_selection[sorted_indices]  # Values in descending order
        covariances_sorted = covariances[sorted_indices]  # Sort covariances to match
        
        plt.subplot(1, 3, rep + 1)
        scatter = plt.scatter(x_vals, y_vals, c=covariances_sorted, cmap='coolwarm', alpha=0.7)
        plt.colorbar(scatter, label='Covariance with Best Dimension')
        plt.title(f'Layer {layer} Replicate {rep + 1}')
        plt.xlabel('Rank of Inferred Selection Coefficient')
        plt.ylabel('Inferred Selection Coefficient (Normalized)')
        plt.axhline(0, color='black', linestyle='--')
        
        
        # For the best dimension highlight:
        best_dim_position = np.where(sorted_indices == best_dim_idx)[0]
        plt.scatter(best_dim_position, normalized_selection[best_dim_idx],
                    color='yellow', edgecolor='black', s=100, label='Best Dimension')
        
    
    plt.tight_layout()
    plt.show()
        
        
    
    

In [ ]:
import numpy as np
from scipy.stats import rankdata
import matplotlib.pyplot as plt


for layer in sim_results[2].keys():
    layer_df = get_df_selection(random_seed=42, selected_layer=layer,
                                 normalize_embeddings=False)[0]
    layer_df = normalize_embeddings(layer_df)
    embeddings = np.vstack(layer_df['Embedding'].values)
    
    # Get the selection coefficients
    true_selection = sim_results[1][layer]
    best_dim_idx = np.argmax(np.abs(true_selection))
    
    # Find the covariance between all embeddings with the best dimension
    best_dim_values = embeddings[:, best_dim_idx]
    
    
    
    """covariances = []
    for dim in range(embeddings.shape[1]):
        dim_values = embeddings[:, dim]
        covariance = np.cov(best_dim_values, dim_values)[0, 1]
        covariances.append(covariance)
    covariances = np.array(covariances)"""
    
    # Find the covariance using a weighted approach and the population from the first selection event (gen=1)
    weights = np.average(all_generation_counts[layer][1], axis=0)
    covariances = []
    for dim in range(embeddings.shape[1]):
        dim_values = embeddings[:, dim]
        mean_best = np.average(best_dim_values, weights=weights)
        mean_dim = np.average(dim_values, weights=weights)
        covariance = np.average((best_dim_values - mean_best) * (dim_values - mean_dim), weights=weights)
        covariances.append(covariance)
    covariances = np.array(covariances)
    
    # Plot the selection coefficients in order of rank, colored by covariance with best dimension
    # Plot all three replicates
    plt.figure(figsize=(15, 5))
    plt.style.use('seaborn-v0_8-darkgrid')
    for rep in range(3):
        inferred_selection = sim_results[2][layer][0][rep]
        
        normalized_selection = z_normalize(inferred_selection)
        
        # Use this:
        sorted_indices = np.argsort(normalized_selection)[::-1]  # Sort descending
        x_vals = np.arange(len(normalized_selection))  # Simple 0, 1, 2, ... for x-axis
        y_vals = normalized_selection[sorted_indices]  # Values in descending order
        covariances_sorted = covariances[sorted_indices]  # Sort covariances to match
        
        plt.subplot(1, 3, rep + 1)
        scatter = plt.scatter(x_vals, y_vals, c=covariances_sorted, cmap='coolwarm', alpha=0.7)
        plt.colorbar(scatter, label='Covariance with Best Dimension')
        plt.title(f'Layer {layer} Replicate {rep + 1}')
        plt.xlabel('Rank of Inferred Selection Coefficient')
        plt.ylabel('Inferred Selection Coefficient (Normalized)')
        plt.axhline(0, color='black', linestyle='--')
        
        
        # For the best dimension highlight:
        best_dim_position = np.where(sorted_indices == best_dim_idx)[0]
        plt.scatter(best_dim_position, normalized_selection[best_dim_idx],
                    color='yellow', edgecolor='black', s=100, label='Best Dimension')
        
    
    plt.tight_layout()
    plt.show()
        
        
    
    

In [ ]:
import numpy as np
from scipy.stats import rankdata
import matplotlib.pyplot as plt


layer=22
layer_df = get_df_selection(random_seed=42, selected_layer=layer,
                                normalize_embeddings=False)[0]
layer_df = normalize_embeddings(layer_df)
embeddings = np.vstack(layer_df['Embedding'].values)

# Get the selection coefficients
true_selection = sim_results[1][layer]
best_dim_idx = np.argmax(np.abs(true_selection))

# Find the covariance between all embeddings with the best dimension
best_dim_values = embeddings[:, best_dim_idx]



"""covariances = []
for dim in range(embeddings.shape[1]):
    dim_values = embeddings[:, dim]
    covariance = np.cov(best_dim_values, dim_values)[0, 1]
    covariances.append(covariance)
covariances = np.array(covariances)"""
for gen in range(50):
    print(f"Generation: {gen}")
    # Find the covariance using a weighted approach and the population from the first selection event (gen=1)
    weights = np.average(all_generation_counts[layer][gen], axis=0)
    covariances = []
    for dim in range(embeddings.shape[1]):
        dim_values = embeddings[:, dim]
        mean_best = np.average(best_dim_values, weights=weights)
        mean_dim = np.average(dim_values, weights=weights)
        covariance = np.average((best_dim_values - mean_best) * (dim_values - mean_dim), weights=weights)
        covariances.append(covariance)
    covariances = np.array(covariances)

    # Plot the selection coefficients in order of rank, colored by covariance with best dimension
    # Plot all three replicates
    plt.figure(figsize=(15, 5))
    plt.style.use('seaborn-v0_8-darkgrid')
    for rep in range(3):
        inferred_selection = sim_results[2][layer][0][rep]
        
        normalized_selection = z_normalize(inferred_selection)
        
        # Use this:
        sorted_indices = np.argsort(normalized_selection)[::-1]  # Sort descending
        x_vals = np.arange(len(normalized_selection))  # Simple 0, 1, 2, ... for x-axis
        y_vals = normalized_selection[sorted_indices]  # Values in descending order
        covariances_sorted = covariances[sorted_indices]  # Sort covariances to match
        
        plt.subplot(1, 3, rep + 1)
        scatter = plt.scatter(x_vals, y_vals, c=covariances_sorted, cmap='coolwarm', alpha=0.7)
        plt.colorbar(scatter, label='Covariance with Best Dimension')
        plt.title(f'Layer {layer} Replicate {rep + 1}')
        plt.xlabel('Rank of Inferred Selection Coefficient')
        plt.ylabel('Inferred Selection Coefficient (Normalized)')
        plt.axhline(0, color='black', linestyle='--')
        
        
        # For the best dimension highlight:
        best_dim_position = np.where(sorted_indices == best_dim_idx)[0]
        plt.scatter(best_dim_position, normalized_selection[best_dim_idx],
                    color='yellow', edgecolor='black', s=100, label='Best Dimension')
        

    plt.tight_layout()
    plt.show()
    
    

    

In [ ]:
# Now Plot the Change in Embedding against the selection coefficients
for layer in range(n_layers):
    inferred_selection = new_detailed_selection_results[layer][-1][chosen_replicate]
    normalized_selection = z_normalize(inferred_selection)
    true_selection = new_all_selection_coefficients[layer]
    
    
    total_change = avg_embeddings[layer, -1, :] - avg_embeddings[layer, 0, :]
    
    
    
    print(f"Normalized_selection Change shape: {normalized_selection.shape}, Total_change shape: {total_change.shape}")
    # Plot the change in embedding vs selection coefficient
    plt.figure(figsize=(8, 6))
    plt.style.use('seaborn-v0_8-darkgrid')
    
    # Color by the true selection
    
    
    plt.scatter(normalized_selection, total_change, alpha=0.5,
                c=true_selection, cmap='viridis')
    plt.title(f'Layer {layer} Change in Embedding vs Selection Coefficient')
    plt.xlabel('Normalized Selection Coefficient')
    plt.ylabel('Change in Embedding Value')
    plt.axhline(0, color='red', linestyle='--')
    plt.axvline(0, color='red', linestyle='--')
    plt.tight_layout()
    plt.show()
    

## With the normal plus one fitness function

In [ ]:
"""def generate_one_selection(embeddings):
    selection_coefficients = np.zeros(embeddings.shape[1])
    embedding_ranges = embeddings.max(axis=0) - embeddings.min(axis=0)
    # Find the indices of the top, lowest, and middle range dimensions
    sorted_indices = np.argsort(embedding_ranges)
    high_range_idx = sorted_indices[-1]
    # Give these dimensions higher selection coefficients
    selection_coefficients[high_range_idx] = 0.10
    return selection_coefficients

def get_simulation_results(n_layers, n_gens, n_reps, 
                           sel_func=generate_selection,
                           inference=True, fitness='plus1',
                           save_every=1):
    all_layer_fits = {}
    all_selection_coefficients = {}
    detailed_selection_results = {}
    all_generation_counts = {}

    for layer in range(n_layers):
        print(f"Running layer {layer}...")
        df_selection, initial_counts = get_df_selection(random_seed=42, selected_layer=layer,
                                                        normalize_embeddings=False)
        df_selection = normalize_embeddings(df_selection)

        # Generate selection coefficients using the provided function
        embedding_matrix = np.vstack(df_selection['Embedding'].values)
        selection_coefficients = sel_func(embedding_matrix)
        all_selection_coefficients[layer] = selection_coefficients
        
        print("Running simulation...")
        generation_counts, layer_fits = run_simulation(df_selection, selection_coefficients, 
                                           initial_counts, n_gens=n_gens, save_every=save_every,
                                           fitness=fitness)
        all_generation_counts[layer] = generation_counts
        all_layer_fits[layer] = layer_fits

        if inference:
            gen=n_gens
            print(f"  Analyzing generation {gen}...")
            layer_results = []
            test_path = pwd + f"/simulations/layer_{layer}_gen_{gen}/"
            data = run_inference_calcs_sims(df_selection, generation_counts[:gen + 1], test_path)
            found_sel_coeffs = data[2]
            layer_results.append(found_sel_coeffs)

            detailed_selection_results[layer] = layer_results

    return all_layer_fits, all_selection_coefficients, detailed_selection_results, all_generation_counts, embedding_matrix
    
    
"""

In [ ]:
one_plus_data = get_simulation_results(n_layers=31, n_gens=50, n_reps=3,
                                        sel_func=generate_selection,
                                        inference=False)


In [ ]:
# plot fitness over time
def plot_fitness_over_time(sim_data):
    # For each layer, plto the fitness growth over time
    
    all_layer_fits = sim_data[0]
    all_gen_counts = sim_data[3]
    
    n_gens = len(all_gen_counts[0]) - 1
    n_reps = len(all_gen_counts[0][0])
    
    avg_fitness_over_time = {}
    for layer in range(len(all_layer_fits.keys())):
        layer_fits = all_layer_fits[layer]
        layer_counts = all_gen_counts[layer]
        avg_fitness_by_rep = []
        for gen in range(len(layer_counts)):
            gen_counts = layer_counts[gen]
            gen_fitnesses = []
            for rep in range(n_reps):
                rep_counts = gen_counts[rep]
                fitnesses = np.array(layer_fits)
                fitnesses = z_normalize(fitnesses)
                fitnesses = fitnesses / np.max(fitnesses)
                avg_fitness = np.sum(fitnesses * rep_counts) / np.sum(rep_counts)
                gen_fitnesses.append(avg_fitness)
            avg_fitness_by_rep.append(gen_fitnesses)
        # Add the replicate information to the layer
        avg_fitness_over_time[layer] = avg_fitness_by_rep

    # Plot the growtih in fitness over time for each layer and replicate
    save_every = 1
    x_vals = np.arange(0, n_gens + 1, save_every)
    
    plt.figure(figsize=(10, 6))
    plt.style.use('seaborn-v0_8-darkgrid')
    for layer in range(1, len(avg_fitness_over_time.keys())):
        layer_avg_fitness = np.array(avg_fitness_over_time[layer])
        for rep in range(n_reps):
            #print(x_vals.shape, layer_avg_fitness[:, rep].shape)
            plt.plot(x_vals, layer_avg_fitness[:, rep], label=f'Layer {layer} Replicate {rep + 1}')
    plt.xlabel('Generation')
    plt.ylabel('Average Fitness')
    plt.title('Average Fitness over Generations for Each Layer and Replicate')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_fitness_over_time(one_plus_data)

In [ ]:
one_plus_data = get_simulation_results(n_layers=31, n_gens=50, n_reps=3,
                                        sel_func=generate_one_selection,
                                        inference=True)


In [ ]:
def plot_inferred_vs_true_sel(sim_data):
    # The selections of every coefficients, and the inferred selections of every coefficient across layer and generation
    detailed_selection_results = sim_data[2]
    all_sel_coeffs = sim_data[1]
    for layer in detailed_selection_results.keys():
        layer_selection = layer
        true_selection = all_sel_coeffs[layer_selection]
        normalized_selection = z_normalize(true_selection)

        for gen in range(len(detailed_selection_results[0])):
            inferred_selection = detailed_selection_results[layer_selection][gen]
            
            fig, axs = plt.subplots(1, 3, figsize=(18, 6))
            plt.style.use('seaborn-v0_8-darkgrid')
            for rep in range(n_reps):
                x = true_selection
                y = z_normalize(inferred_selection[rep])
                
                
                axs[rep].scatter(x, y, alpha=0.5)
                axs[rep].set_title(f'Layer {layer_selection} Generation {gen * 2 + 1} Replicate {rep + 1}')
                axs[rep].set_xlabel('True Selection Coefficients')
                axs[rep].set_ylabel('Inferred Selection Coefficients')
                """axs[rep].plot([min(normalized_selection), max(normalized_selection)],
                            [min(normalized_selection), max(normalized_selection)],
                            color='red', linestyle='--')"""
            
                axs[rep].set_xlim(-0.05, 0.12)
            
            plt.tight_layout()
            plt.show()


In [ ]:
plot_inferred_vs_true_sel(one_plus_data)

In [ ]:
def ordered_cov_plots(sim_data, layer=0):
    layer_df = get_df_selection(random_seed=42, selected_layer=layer,
                                    normalize_embeddings=False)[0]
    layer_df = normalize_embeddings(layer_df)
    embeddings = np.vstack(layer_df['Embedding'].values)

    # Get the selection coefficients
    true_selection = sim_data[1][layer]
    layer_generation_counts = sim_data[3][layer]
    inferred_selection = sim_data[2][layer][0]
    best_dim_idx = np.argmax(np.abs(true_selection))

    # Find the covariance between all embeddings with the best dimension
    best_dim_values = embeddings[:, best_dim_idx]
    
    for gen in range(50):
        print(f"Generation: {gen}")
        # Find the covariance using a weighted approach and the population from the first selection event (gen=1)
        weights = np.average(layer_generation_counts[gen], axis=0)
        covariances = []
        for dim in range(embeddings.shape[1]):
            dim_values = embeddings[:, dim]
            mean_best = np.average(best_dim_values, weights=weights)
            mean_dim = np.average(dim_values, weights=weights)
            covariance = np.average((best_dim_values - mean_best) * (dim_values - mean_dim), weights=weights)
            covariances.append(covariance)
        covariances = np.array(covariances)

        # Plot the selection coefficients in order of rank, colored by covariance with best dimension
        # Plot all three replicates
        plt.figure(figsize=(15, 5))
        plt.style.use('seaborn-v0_8-darkgrid')
        for rep in range(3):
            rep_inf_sel = inferred_selection[rep]
            
            normalized_selection = z_normalize(rep_inf_sel)
            
            # Use this:
            sorted_indices = np.argsort(normalized_selection)[::-1]  # Sort descending
            x_vals = np.arange(len(normalized_selection))  # Simple 0, 1, 2, ... for x-axis
            y_vals = normalized_selection[sorted_indices]  # Values in descending order
            covariances_sorted = covariances[sorted_indices]  # Sort covariances to match
            
            plt.subplot(1, 3, rep + 1)
            scatter = plt.scatter(x_vals, y_vals, c=covariances_sorted, cmap='coolwarm', alpha=0.7)
            plt.colorbar(scatter, label='Covariance with Best Dimension')
            plt.title(f'Layer {layer} Replicate {rep + 1}')
            plt.xlabel('Rank of Inferred Selection Coefficient')
            plt.ylabel('Inferred Selection Coefficient (Normalized)')
            plt.axhline(0, color='black', linestyle='--')
            
            
            # For the best dimension highlight:
            best_dim_position = np.where(sorted_indices == best_dim_idx)[0]
            plt.scatter(best_dim_position, normalized_selection[best_dim_idx],
                        color='yellow', edgecolor='black', s=100, label='Best Dimension')
            

        plt.tight_layout()
        plt.show()
        

In [ ]:
ordered_cov_plots(one_plus_data, layer=22)

## Integrate / average embeding over the generations until fixation

In [ ]:
def find_fixed_gen(generation_counts, cutoff_pct=0.75):
    # Return the generation at which 90% of the population has the same dominant type
    for gen in range(len(generation_counts)):
        for rep in range(len(generation_counts[gen])):
            rep_counts = generation_counts[gen][rep]
            total_count = np.sum(rep_counts)
            max_count = np.max(rep_counts)
            #print(f"Generation: {gen} Replicate: {rep} Max Count: {max_count} Total Count: {total_count}")
            if max_count / total_count >= cutoff_pct:
                return gen
    return len(generation_counts) - 1  # Return the last generation if never reaches cutoff

def averaged_covariance(sim_data, layer=0):
    layer_df = get_df_selection(random_seed=42, selected_layer=layer,
                                    normalize_embeddings=False)[0]
    layer_df = normalize_embeddings(layer_df)
    embeddings = np.vstack(layer_df['Embedding'].values)

    # Get the selection coefficients
    true_selection = sim_data[1][layer]
    layer_generation_counts = sim_data[3][layer]
    inferred_selection = sim_data[2][layer][0]
    best_dim_idx = np.argmax(np.abs(true_selection))

    # Find the covariance between all embeddings with the best dimension
    best_dim_values = embeddings[:, best_dim_idx]
    fixed_gen = find_fixed_gen(layer_generation_counts)
    print(f"Cutoff generation for layer {layer}: {fixed_gen}")
    total_covariances = []
    for gen in range(fixed_gen):
        # Find the covariance using a weighted approach and the population from the first selection event (gen=1)
        weights = np.average(layer_generation_counts[gen], axis=0)
        covariances = []
        for dim in range(embeddings.shape[1]):
            dim_values = embeddings[:, dim]
            mean_best = np.average(best_dim_values, weights=weights)
            mean_dim = np.average(dim_values, weights=weights)
            covariance = np.average((best_dim_values - mean_best) * (dim_values - mean_dim), weights=weights)
            covariances.append(covariance)
        covariances = np.array(covariances)
        total_covariances.append(covariances)
    avg_covariances = np.mean(total_covariances, axis=0)
    # Plot the selection coefficients in order of rank, colored by covariance with best dimension
    # Plot all three replicates
    plt.figure(figsize=(15, 5))
    plt.style.use('seaborn-v0_8-darkgrid')
    for rep in range(3):
        rep_inf_sel = inferred_selection[rep]
        
        normalized_selection = z_normalize(rep_inf_sel)
        
        # Use this:
        sorted_indices = np.argsort(normalized_selection)[::-1]  # Sort descending
        x_vals = np.arange(len(normalized_selection))  # Simple 0, 1, 2, ... for x-axis
        y_vals = normalized_selection[sorted_indices]  # Values in descending order
        covariances_sorted = avg_covariances[sorted_indices]  # Sort covariances to match
        
        plt.subplot(1, 3, rep + 1)
        scatter = plt.scatter(x_vals, y_vals, c=covariances_sorted, cmap='coolwarm', alpha=0.7)
        plt.colorbar(scatter, label='Average Covariance with Best Dimension')
        plt.title(f'Layer {layer} Replicate {rep + 1}')
        plt.xlabel('Rank of Inferred Selection Coefficient')
        plt.ylabel('Inferred Selection Coefficient (Normalized)')
        plt.axhline(0, color='black', linestyle='--')
        
        
        # For the best dimension highlight:
        best_dim_position = np.where(sorted_indices == best_dim_idx)[0]
        plt.scatter(best_dim_position, normalized_selection[best_dim_idx],
                    color='yellow', edgecolor='black', s=100, label='Best Dimension')
        
    plt.tight_layout()
    plt.show()

        

In [ ]:
for layer in range(31):
    averaged_covariance(one_plus_data, layer=layer)

In [ ]:
plot_fitness_over_time(one_plus_data)

In [ ]:
one_plus_data

## Looking at inferred vs simulated fitness 

In [ ]:
one_plus_data = get_simulation_results(n_layers=31, n_gens=50, n_reps=3,
                                        sel_func=generate_one_selection,
                                        inference=True, fitness='plus1',
                                        save_every=1)

In [ ]:
comp_inf_vs_real_fits(one_plus_data, layer=22, fitness='plus1')

In [ ]:
for layer in range(31):
    comp_inf_vs_real_fits(one_plus_data, layer=layer, fitness='plus1')

In [ ]:
for layer in range(31):
    comp_inf_vs_real_fits_rank(one_plus_data, layer=layer, fitness='plus1')

In [ ]:
pwd = os.getcwd()
sim_folder = pwd + "/esm_sim_saves/"

def save_sim_data(sim_data, filename):
    if not os.path.exists(sim_folder):
        os.makedirs(sim_folder)
    with open(sim_folder + filename, 'wb') as f:
        pickle.dump(sim_data, f)
        
        
def load_sim_data(filename):
    with open(sim_folder + filename, 'rb') as f:
        sim_data = pickle.load(f)
    return sim_data

In [ ]:
save_sim_data(one_plus_data, "one_plus_data_one_sel.pkl")

In [ ]:
def gaussian_selection(embeddings):
    width = 0.02
    center = 0.0
    sel_coeffs = np.random.normal(loc=center, scale=width, size=embeddings.shape[1])
    return sel_coeffs

In [ ]:
one_plus_multi_sel_data = get_simulation_results(n_layers=31, n_gens=50, n_reps=3,
                                        sel_func=gaussian_selection,
                                        inference=True, fitness='plus1',
                                        save_every=1)

In [ ]:
save_sim_data(one_plus_multi_sel_data, "one_plus_data_gaussian_sel.pkl")

In [ ]:
one_plus_multi_sel_data = load_sim_data("one_plus_data_gaussian_sel.pkl")

In [ ]:
for layer in range(31):
    comp_inf_vs_real_fits(one_plus_data, layer=layer, fitness='plus1')

In [ ]:
for layer in range(31):
    comp_inf_vs_real_fits_rank(one_plus_data, layer=layer, fitness='plus1')

In [43]:
def zero_selection(embeddings):
    return np.zeros(embeddings.shape[1])

In [ ]:
one_plus_zero_sel_data = get_simulation_results(n_layers=31, n_gens=50, n_reps=3,
                                        sel_func=zero_selection,
                                        inference=True, fitness='plus1',
                                        save_every=1)

In [ ]:
save_sim_data(one_plus_zero_sel_data, "one_plus_data_zero_sel.pkl")

In [ ]:
for layer in range(31):
    comp_inf_vs_real_fits(one_plus_zero_sel_data, layer=layer, fitness='plus1')

In [ ]:
for layer in range(31):
    comp_inf_vs_real_fits_rank(one_plus_zero_sel_data, layer=layer, fitness='plus1')

# Simulating BF and BG together!!

In [45]:
def gaussian_selection(embeddings):
    width = 0.02
    center = 0.0
    sel_coeffs = np.random.normal(loc=center, scale=width, size=embeddings.shape[1])
    return sel_coeffs

def zero_selection(embeddings):
    return np.zeros(embeddings.shape[1])

def generate_selection(embeddings):
    embedding_ranges = embeddings.max(axis=0) - embeddings.min(axis=0)
    # Find the indices of the top, lowest, and middle range dimensions
    selection_coefficients = np.zeros(embeddings.shape[1])
    sorted_indices = np.argsort(embedding_ranges)
    high_range_idx = sorted_indices[-1]
    # Give these dimensions higher selection coefficients
    selection_coefficients[high_range_idx] = 0.10
    return selection_coefficients

In [66]:
## DEFINITIONS (DIRTY NOW, CLEAN UP LATER)
def get_layer_df_piecewise(layer, in_paths, normalize=True):
    """Get the combined layer_df from multiple input sources, together"""
    dfs = []
    for in_path in in_paths:
        df = pickle.load(open(f"{in_path}/layer{layer}/inference_df.pkl", 'rb'))
        dfs.append(df) 
    layer_df = dfs[0].copy()
    num_reps = len(layer_df["Replicate"].unique())
    
    """print(layer_df.columns)"""
    
    total_paths = len(in_paths)
    for path_idx in range(1, total_paths):
        df_to_add = dfs[path_idx].copy()
        df_to_add["Replicate"] = df_to_add["Replicate"] + path_idx * num_reps
        layer_df = pd.concat([layer_df, df_to_add], ignore_index=True)

    if normalize:
        embeddings = np.array([x for x in layer_df["Embedding"].to_list()])
        dimensions = embeddings.shape[1]
        for dim in range(dimensions):
            embeddings[:, dim] = z_normalize(embeddings[:, dim])
        layer_df["Embedding"] = [embeddings[i] for i in range(embeddings.shape[0])]

    return layer_df

def convert_long_to_wide(df):
    """Convert layer_df from long format to wide format.
    
    Before: one row per (Embedding, Replicate, Generation) with a Frequency column.
    After:  one row per unique Embedding with columns Rep{i}_PreNums / Rep{i}_PostNums.
    
    Generation 0 -> PreNums, Generation 1 -> PostNums.
    Replicates are 0-indexed in the input and 1-indexed in the output.
    """
    gen_map = {0: 'PreNums', 1: 'PostNums'}
    # Use tuple as a hashable embedding key
    df = df.copy()
    df['_emb_key'] = df['Embedding'].apply(tuple)
    # Pivot Frequency into (Replicate, Generation) columns
    wide = df.pivot_table(
        index='_emb_key',
        columns=['Replicate', 'Generation'],
        values='Frequency',
        aggfunc='sum'
    ).fillna(0)
    # Flatten and rename columns: (rep, gen) -> Rep{rep+1}_{PreNums|PostNums}
    wide.columns = [
        f'Rep{rep}_{gen_map[gen]}'
        for rep, gen in wide.columns
    ]
    wide = wide.reset_index()
    # Restore numpy arrays and drop the temp key
    wide['Embedding'] = wide['_emb_key'].apply(np.array)
    wide = wide.drop(columns='_emb_key')
    # Reorder: all Pre/Post columns first, then Embedding
    rep_cols = [c for c in wide.columns if c != 'Embedding']
    wide = wide[rep_cols + ['Embedding']].reset_index(drop=True)
    return wide

def get_simulation_results_piecewise(n_layers, n_gens, n_reps,
                                    in_paths, 
                                    sel_func=generate_selection,
                                    inference=True, fitness='plus1',
                                    save_every=1):
    all_layer_fits = {}
    all_selection_coefficients = {}
    detailed_selection_results = {}
    all_generation_counts = {}

    
    
    for layer in range(n_layers):
        print(f"Running layer {layer}...")
        
        df_layer = get_layer_df_piecewise(layer, in_paths, normalize=True)
        df_selection = convert_long_to_wide(df_layer)
        
        n_reps = len(df_selection.columns) // 2  # Assuming each replicate has PreNums and PostNums
        
        initial_counts = []
        for rep in range(n_reps):
            pre_col = f'Rep{rep + 1}_PreNums'
            if pre_col not in df_selection.columns:
                raise ValueError(f"Expected column {pre_col} not found in df_selection: {df_selection.columns}")
            initial_counts.append(df_selection[pre_col].values)
        

        # Generate selection coefficients using the provided function
        embedding_matrix = np.vstack(df_selection['Embedding'].values)
        selection_coefficients = sel_func(embedding_matrix)
        all_selection_coefficients[layer] = selection_coefficients
        
        print("Running simulation...")
        generation_counts, layer_fits = run_simulation(df_selection, selection_coefficients, 
                                           initial_counts, n_gens=n_gens, save_every=save_every,
                                           fitness=fitness)
        all_generation_counts[layer] = generation_counts
        all_layer_fits[layer] = layer_fits

        if inference:
            gen=n_gens
            print(f"  Analyzing generation {gen}...")
            layer_results = []
            test_path = pwd + f"/simulations/layer_{layer}_gen_{gen}/"
            data = run_inference_calcs_sims(df_selection, generation_counts[:gen + 1], test_path)
            found_sel_coeffs = data[2]
            layer_results.append(found_sel_coeffs)

            detailed_selection_results[layer] = layer_results

    return all_layer_fits, all_selection_coefficients, detailed_selection_results, all_generation_counts, embedding_matrix
    

In [23]:
bg_path = pwd + "/data/inference_data/BG505"
bf_path = pwd + "/data/inference_results"

In [67]:
combined_sim_data = get_simulation_results_piecewise(31, 50, 3, [bg_path, bf_path], 
                                                     sel_func=gaussian_selection,
                                                        inference=True, fitness='plus1',
                                                        save_every=1)

Running layer 0...
Running simulation...
[array([16., 26., 28., ...,  0.,  0.,  0.], shape=(26368,)), array([15., 26., 28., ...,  0.,  0.,  0.], shape=(26368,)), array([13., 25., 23., ...,  0.,  0.,  0.], shape=(26368,)), array([ 0.,  0.,  0., ..., 18., 30., 22.], shape=(26368,)), array([ 0.,  0.,  0., ..., 37., 31., 24.], shape=(26368,)), array([ 0.,  0.,  0., ..., 31., 21., 36.], shape=(26368,))]
  Analyzing generation 50...


In [ ]:
plot_fitness_over_time(combined_sim_data)

In [ ]:
for layer in range(31):
    comp_inf_vs_real_fits(combined_sim_data, layer=layer, fitness='plus1')
    


In [ ]:
for layer in range(31):
    comp_inf_vs_real_fits_rank(combined_sim_data, layer=layer, fitness='plus1')

In [ ]:
for layer in range(31):
    averaged_covariance(combined_sim_data, layer=layer)

In [ ]:
for layer in range(31):
    ordered_cov_plots(combined_sim_data, layer=layer)

In [ ]:
plot_inferred_vs_true_sel(combined_sim_data)

In [1]:
df_selection, initial_counts = get_df_selection(random_seed=42, selected_layer=layer)
embedding_matrix = np.vstack(df_selection['Embedding'].values)

NameError: name 'get_df_selection' is not defined